# V4.4.0 Seven Class: Fresh Training Workflow

This notebook builds an experimental seven-class mango model from the raw image collection. It does not load an earlier mango model, split, mask, run folder, or threshold file. ImageNet weights supply the starting visual features; all mango-specific training and records are new.

The seven classes are Anthracnose, Bacterial Black Spot, Bacterial Canker, Healthy Fruit, Healthy Leaf, Mango Scab, and Sooty Mold. The planned training is five head-training epochs and ten fine-tuning epochs. An epoch is one pass through the training examples.

Run the numbered steps in order. Review gates intentionally stop until the image sheets have been inspected and the decisions recorded. A similarity warning does not prove that two photographs are duplicates. Keep related photographs in one subset so evaluation does not benefit from a similar training photograph.

Outputs stay under `My Drive/mangoleaf_experiments/v4.4.0/training/<unique-run-id>/`. This location does not imply inheritance. The application continues to use its adopted model until this candidate is separately evaluated and adopted.

This template has no saved training outputs. Do not report an accuracy, selected epoch, temperature, rejection rate, or production deployment until the corresponding run has produced that evidence.

## Step 1: Connect Drive and check the software

**What:** Mount Google Drive and prepare separate setup, scientific, and converter environments.

**Why:** Colab contains many provider libraries unrelated to this experiment. The actual training packages must be checked without replacing the provider's kernel libraries.

**How:** The first two cells run in Colab: mount Drive, create an empty setup environment without ensurepip, use pip --python to install only into that environment, record host findings, install audited isolated packages, and check real GPU execution and conversion. Provider pip must support --python (version 22.3 or newer). All remaining code cells begin with %%mangoleaf; this marker sends visible code to the same isolated Python process and returns logs and figures to Colab. No extra activation commands or dataset uploads are needed.

**Expected result:** The isolated interpreter path is printed and runtime evidence is saved. Host findings remain visible; unreviewed worker advisories, skipped worker packages, dependency conflicts, GPU failures, or conversion failures stop the workflow. This does not establish browser parity or model accuracy.

**Kernel startup:** The scientific process has a private IPython profile. Colab's configuration and automatic startup scripts are not loaded into that process. A startup failure names a local `kernel-startup.log`; connection credentials stay outside Drive evidence.

If environment preparation completed in the current session, it can be reused after a launcher repair: run the updated helper-definition cell, then run `start_notebook_kernel(RUNTIME_REVIEW)` in a new ordinary Colab code cell. Package preparation is not repeated. After startup succeeds, continue with the scientific-import cell below.

In [ ]:
"""Prepare separate audited Python packages without changing Colab libraries.

Colab mounts Drive and forwards notebook output. Scientific cells execute in a
persistent isolated kernel; package isolation is not a separate security sandbox.
"""

import hashlib
import importlib.metadata
import json
import os
import site
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path


# New setup choices. Historical full Colab package snapshots are never installed.
SECURITY_PACKAGES = (
    "pip==26.2.1", "setuptools==84.0.0", "requests==2.34.2",
    "urllib3==2.8.0", "pip-audit==2.10.1",
)
TOOL_PACKAGES = (*SECURITY_PACKAGES, "uv==0.12.23")
TRAINING_PACKAGES = (
    *SECURITY_PACKAGES,
    "tensorflow[and-cuda]==2.19.0", "keras==3.15.1", "numpy==2.1.3",
    "Pillow==12.3.0", "ImageHash==4.3.2", "scipy==1.16.3",
    "matplotlib==3.11.2", "scikit-learn==1.9.1",
    "opencv-python-headless==5.0.0.93", "ipykernel==7.4.0",
    "ipython==9.17.1", "jupyter-client==8.10.0", "tornado==6.5.10",
)
CONVERTER_PACKAGES = (
    *(package for package in SECURITY_PACKAGES if not package.startswith("setuptools==")),
    # TensorFlow Hub 0.16.1 needs the real pkg_resources API removed in 82.
    # Its sdist-only advisory stays visible and gets a narrow operation review.
    "setuptools==81.0.0",
    "wheel==0.45.1",
    "tensorflow==2.19.0", "tf-keras==2.19.0",
    "tensorflow-decision-forests==1.12.0", "tensorflow-hub==0.16.1",
    # TFDF accepts newer YDF releases whose generated protos require Protobuf 6.
    # TensorFlow 2.19 requires Protobuf below 6; YDF 0.11 uses gencode 5.29.1.
    "ydf==0.11.0", "protobuf==5.29.6",
    "tensorflowjs==4.22.0", "keras==3.15.1", "Pillow==12.3.0",
    "numpy==2.1.3", "jax==0.4.35", "jaxlib==0.4.35",
    "flax==0.10.2", "packaging==23.2",
)


def runtime_write_json_once(path, value):
    """Preserve each attempt instead of overwriting earlier evidence."""
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write("\n")


def runtime_package_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result


def runtime_clean_environment():
    """Prevent host Python paths and legacy Keras settings from leaking in."""
    environment = dict(os.environ)
    for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "TF_USE_LEGACY_KERAS"):
        environment.pop(key, None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["TF_CPP_MIN_LOG_LEVEL"] = "2"
    return environment


def runtime_run_checked(command, log_path, environment=None):
    """Save full diagnostics. Never print credentials or a complete environment."""
    print("Runtime check:", Path(log_path).name, flush=True)
    result = subprocess.run(command, check=False, capture_output=True, text=True, env=environment)
    with Path(log_path).open("x", encoding="utf-8") as handle:
        handle.write(result.stdout or "")
        handle.write("\n")
        handle.write(result.stderr or "")
    if result.returncode != 0:
        diagnostic = Path(log_path)
        # A child preflight can fail in its own subprocess. Forward that named
        # log instead of making the outer traceback hide the actual failure.
        prefix = "RuntimeError: Runtime preparation stopped. Inspect the saved diagnostic log: "
        for line in reversed((result.stderr or "").splitlines()):
            if line.startswith(prefix):
                child = Path(line[len(prefix):])
                if child.parent.resolve() == diagnostic.parent.resolve() and child.is_file():
                    diagnostic = child
                break
        raise RuntimeError("Runtime preparation stopped. Inspect the saved diagnostic log: " + str(diagnostic))
    return result


def runtime_audit(python_path, output_directory, name, environment=None, paths=None, enforce=True, converter_build_review=False):
    """Worker findings block execution; provider findings remain explicit evidence."""
    report_path = Path(output_directory) / (name + "-audit.json")
    command = [str(python_path), "-m", "pip_audit", "--format", "json", "--output", str(report_path), "--progress-spinner", "off"]
    for path in paths or []:
        command.extend(["--path", str(path)])
    result = subprocess.run(command, check=False, capture_output=True, text=True, env=environment)
    with (Path(output_directory) / (name + "-audit.log")).open("x", encoding="utf-8") as handle:
        handle.write(result.stdout or "")
        handle.write("\n")
        handle.write(result.stderr or "")

    error = None
    dependencies = []
    try:
        report = json.loads(report_path.read_text(encoding="utf-8"))
        dependencies = report.get("dependencies", [])
        if not isinstance(dependencies, list) or not dependencies or not all(isinstance(row, dict) and row.get("name") for row in dependencies):
            raise ValueError("No complete installed-package inventory was returned.")
    except (OSError, ValueError, AttributeError) as failure:
        dependencies = []
        error = str(failure)

    findings = [{"name": row["name"], "version": row.get("version"), "vulnerabilities": row.get("vulns", [])} for row in dependencies if row.get("vulns")]
    skipped = [row for row in dependencies if row.get("skip_reason")]
    clean = not (error or findings or skipped or result.returncode != 0)
    reviewed = []
    blocking = []
    build_ids = {"GHSA-h35f-9h28-mq5c", "CVE-2026-59890", "PYSEC-2026-3447", "BIT-setuptools-2026-59890"}
    unpack_ids = {"GHSA-8rrh-rw8j-w5fx", "CVE-2026-24049"}
    for finding in findings:
        identities = [set([vulnerability.get("id"), *vulnerability.get("aliases", [])]) for vulnerability in finding["vulnerabilities"]]
        build_only = (
            converter_build_review and name == "converter"
            and finding["name"] == "setuptools" and finding["version"] == "81.0.0"
            and identities and all(identity and identity <= build_ids for identity in identities)
        )
        unpack_only = (
            converter_build_review and name == "converter"
            and finding["name"] == "wheel" and finding["version"] == "0.45.1"
            and identities and all(identity and identity <= unpack_ids for identity in identities)
        )
        if build_only:
            reviewed.append({
                "package": "setuptools", "version": "81.0.0",
                "advisory": "GHSA-h35f-9h28-mq5c",
                "status": "unresolved; affected sdist build/publish operation is absent",
                "reason": "The converter imports pkg_resources, installs wheels only, and converts SavedModels. It does not build or publish Python source distributions.",
                "source": "https://github.com/advisories/GHSA-h35f-9h28-mq5c",
            })
        elif unpack_only:
            reviewed.append({
                "package": "wheel", "version": "0.45.1",
                "advisory": "GHSA-8rrh-rw8j-w5fx",
                "status": "unresolved; affected wheel.cli.unpack API is absent",
                "reason": "The converter retains wheel 0.45.1. Package installation uses uv's installer, not wheel.cli.unpack. Conversion does not invoke wheel unpack or its vendored setuptools equivalent.",
                "source": "https://github.com/advisories/GHSA-8rrh-rw8j-w5fx",
            })
        else:
            blocking.append(finding)
    allowed = clean or (not error and not skipped and result.returncode == 1 and bool(reviewed) and not blocking)
    summary = {
        "status": "passed" if clean else "unresolved",
        "scope": "execution gate" if enforce else "provider host report; not a worker approval",
        "returnCode": result.returncode, "findings": findings, "skipped": skipped,
        "collectionError": error, "installedPackageCount": len(dependencies),
        "operationReviews": reviewed, "executionAllowed": allowed if enforce else None,
    }
    runtime_write_json_once(Path(output_directory) / (name + "-audit-summary.json"), summary)
    print("Dependency audit:", name, summary["status"], "affected packages:", len(findings), "skipped:", len(skipped), flush=True)
    if enforce and not allowed:
        raise RuntimeError("Dependency review is incomplete or contains advisories. Review the saved findings and install compatible fixes before repeating Step 1: " + str(report_path))
    if not enforce and not clean:
        print("Colab host findings remain unresolved. Report:", report_path, flush=True)
    if enforce and reviewed:
        print("Converter retains documented findings for unused sdist/unpack operations. Full reports and operation reviews:", report_path, flush=True)
    return summary


def runtime_check_environment(python_path, evidence, name, environment):
    runtime_run_checked([str(python_path), "-m", "pip", "check"], evidence / (name + "-pip-check.log"), environment)
    freeze = runtime_run_checked([str(python_path), "-m", "pip", "freeze", "--all"], evidence / (name + "-freeze.log"), environment)
    (evidence / (name + "-runtime-lock.txt")).write_text(freeze.stdout, encoding="utf-8")
    return runtime_audit(python_path, evidence, name, environment, converter_build_review=(name == "converter"))


def runtime_prepare_tools(environment_root, evidence, environment):
    """Install setup tools into an empty environment without provider ensurepip."""
    tools_root = Path(environment_root) / "tools"

    # Colab's ensurepip can fail before any experiment package is installed.
    # Create only the Python environment, without asking ensurepip to seed pip.
    runtime_run_checked(
        [sys.executable, "-m", "venv", "--without-pip", str(tools_root)],
        Path(evidence) / "tools-environment.log",
        environment,
    )
    tools_python = tools_root / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

    # pip --python can install into an environment that has no pip yet.
    # The explicit interpreter targets tools_root, never Colab's site packages.
    runtime_run_checked(
        [sys.executable, "-m", "pip", "--python", str(tools_python),
         "install", "--only-binary", ":all:", "--upgrade", *TOOL_PACKAGES],
        Path(evidence) / "tools-package-setup.log",
        environment,
    )
    return tools_python


def prepare_reviewed_runtime(output_root, conversion=True):
    """Build real execution environments; never install into the Colab kernel."""
    output_root = Path(output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    identifier = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    evidence = output_root / ("runtime-" + identifier)
    evidence.mkdir(exist_ok=False)
    environment_root = Path("/content/mangoleaf-v440-runtime") / identifier
    environment_root.mkdir(parents=True, exist_ok=False)
    environment = runtime_clean_environment()

    # The small setup environment uses Colab's Python, but not its packages.
    # All subsequent installs target explicit environment Python executables.
    tools_python = runtime_prepare_tools(environment_root, evidence, environment)
    runtime_check_environment(tools_python, evidence, "tools", environment)

    # Audit the actual provider sites with a separate scanner. Keep skipped
    # provider wheels and host conflicts visible instead of upgrading Colab.
    host_paths = sorted({str(Path(path).resolve()) for path in [*site.getsitepackages(), site.getusersitepackages()] if Path(path).is_dir()})
    runtime_write_json_once(evidence / "host-inventory.json", {
        "python": sys.version,
        "paths": host_paths,
        "packages": sorted([{"name": dist.metadata.get("Name"), "version": dist.version} for dist in importlib.metadata.distributions()], key=lambda row: str(row["name"]).lower()),
    })
    host_check = subprocess.run([sys.executable, "-m", "pip", "check"], check=False, capture_output=True, text=True)
    (evidence / "host-pip-check.log").write_text((host_check.stdout or "") + "\n" + (host_check.stderr or ""), encoding="utf-8")
    host_summary = runtime_audit(tools_python, evidence, "host", environment, paths=host_paths, enforce=False)
    runtime_write_json_once(evidence / "host-review.json", {
        "audit": host_summary, "pipCheckReturnCode": host_check.returncode,
        "hostPackagesModified": False,
        "boundary": "Package isolation does not patch or sandbox the Colab host.",
    })

    # TFDF's released requirement is exactly TensorFlow 2.19.0. Python 3.11
    # supports that stack independently of the provider's kernel version.
    environment["UV_CACHE_DIR"] = str(environment_root / "uv-cache")
    environments = {"training": TRAINING_PACKAGES}
    if conversion:
        environments["converter"] = CONVERTER_PACKAGES
    interpreters = {}
    for name, packages in environments.items():
        folder = environment_root / name
        runtime_run_checked([str(tools_python), "-m", "uv", "venv", "--python", "3.11", "--managed-python", "--seed", str(folder)], evidence / (name + "-environment.log"), environment)
        python_path = folder / "bin" / "python"
        runtime_run_checked([str(tools_python), "-m", "uv", "pip", "install", "--only-binary", ":all:", "--python", str(python_path), *packages], evidence / (name + "-package-setup.log"), environment)
        runtime_check_environment(python_path, evidence, name, environment)
        interpreters[name] = python_path

    training_python = interpreters["training"]
    # Use the GPU libraries resolved with tensorflow[and-cuda], not mismatched
    # provider CUDA wheels. The machine's NVIDIA driver remains shared.
    nvidia_root = training_python.parent.parent / "lib/python3.11/site-packages/nvidia"
    cuda_paths = [str(path) for path in sorted(nvidia_root.glob("*/lib")) if path.is_dir()]
    cuda_bins = [str(path) for path in sorted(nvidia_root.glob("*/bin")) if path.is_dir()]
    training_environment = runtime_clean_environment()
    training_environment["LD_LIBRARY_PATH"] = os.pathsep.join([*cuda_paths, training_environment.get("LD_LIBRARY_PATH", "")])
    training_environment["PATH"] = os.pathsep.join([str(training_python.parent), *cuda_bins, training_environment.get("PATH", "")])
    training_environment["MPLBACKEND"] = "module://matplotlib_inline.backend_inline"

    # Both fitting and training prove the interpreter/package boundary first.
    isolation_probe = "import json,sys,site; assert sys.prefix != sys.base_prefix; assert not site.ENABLE_USER_SITE; print(json.dumps({'python':sys.executable,'prefix':sys.prefix,'paths':sys.path}))"
    runtime_run_checked([str(training_python), "-I", "-c", isolation_probe], evidence / "training-isolation-probe.log", training_environment)
    converter_python = interpreters.get("converter")
    if conversion:
        probe_file = evidence / "check-training-export.py"
        import inspect
        probe_source = "import hashlib, importlib.metadata, json, os, subprocess, sys\nfrom pathlib import Path\n"
        for helper in (runtime_write_json_once, runtime_package_versions, runtime_clean_environment, runtime_run_checked, runtime_conversion_probe):
            probe_source += "\n" + inspect.getsource(helper)
        probe_file.write_text(probe_source + "\nruntime_conversion_probe(Path(sys.argv[1]), Path(sys.argv[2]))\n", encoding="utf-8")
        runtime_run_checked([str(training_python), "-I", str(probe_file), str(evidence), str(converter_python)], evidence / "training-export-probe.log", training_environment)

    record = {
        "status": "isolated execution checks completed; inspect retained audit findings and operation reviews",
        "hostReview": "separate provider report; unresolved findings are not worker approval",
        "hostPackagesModified": False,
        "trainingPython": str(training_python), "trainingPackages": list(TRAINING_PACKAGES),
        "managedPythonRequest": "3.11",
        "converterPython": str(converter_python) if converter_python else None,
        "converterPackages": list(CONVERTER_PACKAGES) if conversion else [],
        "preflight": json.loads((evidence / "conversion-probe.json").read_text(encoding="utf-8")) if conversion else {"conversion": "not repeated for threshold fitting"},
        "artifacts": {path.relative_to(evidence).as_posix(): {"sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "bytes": path.stat().st_size} for path in sorted(evidence.rglob("*")) if path.is_file()},
    }
    runtime_write_json_once(evidence / "runtime-preflight-complete.json", record)
    print("Isolated runtime prepared:", training_python, flush=True)
    return {
        "evidenceDirectory": evidence, "trainingPython": training_python,
        "trainingEnvironment": training_environment,
        "converterPython": converter_python,
        "converterCommand": converter_python.parent / "tensorflowjs_converter" if converter_python else None,
        "record": record,
    }


def runtime_conversion_probe(evidence, converter_python):
    """Check real GPU execution and cross-process export before any fitting."""
    evidence = Path(evidence)
    converter_python = Path(converter_python)
    converter_root = converter_python.parent.parent
    converter_environment = runtime_clean_environment()
    converter_environment.update(TF_USE_LEGACY_KERAS="1", CUDA_VISIBLE_DEVICES="-1")
    import numpy as np
    import tensorflow as tf

    # This uses the actual architecture but no pretrained weights and no fitting.
    # Conversion support must be checked before spending time on 15 training epochs.
    tf.keras.utils.set_random_seed(42)
    backbone = tf.keras.applications.MobileNetV3Large(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None,
        include_preprocessing=True,
    )
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(224, 224, 3), dtype="float32", name="input_image")
    features = backbone(inputs, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D(name="gap")(features)
    features = tf.keras.layers.Dropout(0.2, name="dropout")(features)
    logits = tf.keras.layers.Dense(7, name="logits")(features)
    outputs = tf.keras.layers.Activation("softmax", dtype="float32", name="probs")(
        logits
    )
    probe_model = tf.keras.Model(inputs, outputs, name="mangoleaf_v4_4_0_preflight")
    pixels = np.random.default_rng(42).uniform(
        0.0, 255.0, size=(1, 224, 224, 3)
    ).astype(np.float32)
    expected = probe_model(pixels, training=False).numpy()
    np.save(evidence / "preflight-input.npy", pixels, allow_pickle=False)
    np.save(evidence / "preflight-training-output.npy", expected, allow_pickle=False)
    saved_model = evidence / "preflight-saved-model"
    probe_model.export(str(saved_model), format="tf_saved_model", verbose=False)

    worker = evidence / "check-converter-saved-model.py"
    worker.write_text(
        "import sys\n"
        "from pathlib import Path\n"
        "import numpy as np\n"
        "import tensorflow as tf\n"
        "folder = Path(sys.argv[1])\n"
        "model = tf.saved_model.load(str(folder / 'preflight-saved-model'))\n"
        "pixels = np.load(folder / 'preflight-input.npy', allow_pickle=False)\n"
        "result = model.signatures['serving_default'](tf.constant(pixels))\n"
        "if len(result) != 1:\n"
        "    raise RuntimeError('Expected exactly one probability output.')\n"
        "probabilities = next(iter(result.values())).numpy()\n"
        "np.save(folder / 'preflight-converter-output.npy', probabilities, "
        "allow_pickle=False)\n",
        encoding="utf-8",
    )
    runtime_run_checked(
        [str(converter_python), str(worker), str(evidence)],
        evidence / "saved-model-compatibility.log",
        converter_environment,
    )
    actual = np.load(evidence / "preflight-converter-output.npy", allow_pickle=False)
    np.testing.assert_allclose(expected, actual, rtol=1e-4, atol=1e-5)

    graph_folder = evidence / "preflight-tfjs-model"
    runtime_run_checked(
        [
            str(converter_root / "bin" / "tensorflowjs_converter"),
            "--input_format=tf_saved_model",
            "--output_format=tfjs_graph_model",
            "--signature_name=serving_default",
            "--saved_model_tags=serve",
            str(saved_model),
            str(graph_folder),
        ],
        evidence / "preflight-conversion.log",
        converter_environment,
    )
    graph = json.loads((graph_folder / "model.json").read_text(encoding="utf-8"))
    if graph.get("format") != "graph-model" or not graph.get("weightsManifest"):
        raise RuntimeError("Converter did not produce a complete graph model.")

    for group in graph["weightsManifest"]:
        for relative in group["paths"]:
            shard = (graph_folder / relative).resolve()
            if not shard.is_relative_to(graph_folder.resolve()) or not shard.is_file():
                raise RuntimeError("A converter weight shard is missing or unsafe.")

        for weight in group["weights"]:
            if weight.get("quantization"):
                raise RuntimeError("The preflight graph unexpectedly uses quantization.")


    gpu_devices = tf.config.list_physical_devices("GPU")
    if not gpu_devices:
        raise RuntimeError("Select a Colab GPU runtime, then repeat Step 1. The isolated training process could not access a GPU.")
    with tf.device("/GPU:0"):
        gpu_check = tf.matmul(tf.ones((2, 2)), tf.ones((2, 2)))
    if "GPU" not in gpu_check.device.upper():
        raise RuntimeError("The training GPU check unexpectedly used a CPU.")

    runtime_write_json_once(evidence / "conversion-probe.json", {
        "trainingPython": sys.executable,
        "trainingPrefix": sys.prefix,
        "trainingVersions": runtime_package_versions(("tensorflow", "keras", "numpy", "Pillow", "scipy")),
        "gpuCount": len(gpu_devices),
        "gpuOperationDevice": gpu_check.device,
        "weights": "random initialization; weights=None",
        "trainingEpochs": 0,
        "savedModelMaximumAbsoluteDifference": float(np.max(np.abs(expected - actual))),
        "tfjsConversionCompleted": True,
        "browserProbabilityParity": "not measured by this preflight",
        "accuracy": "not measured by this preflight",
    })
    tf.keras.backend.clear_session()


"""Forward visible notebook cells to one persistent isolated Python kernel."""

import atexit
import json
import os
import sys
import uuid
from pathlib import Path
from queue import Empty

from IPython.display import clear_output, display
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager


def notebook_kernel_launcher():
    """Use the real IPython kernel with only this run's private configuration."""
    return '''from ipykernel.kernelapp import IPKernelApp


class MangoLeafKernelApp(IPKernelApp):
    def init_config_files(self):
        # IPython normally adds system/provider configuration directories.
        # This fresh profile belongs only to the prepared scientific process.
        self.config_file_paths = [self.profile_dir.location]

    def _run_startup_files(self):
        # The experiment has no automatic startup scripts. IPython normally
        # also searches system directories, even with a private IPYTHONDIR.
        return


if __name__ == "__main__":
    MangoLeafKernelApp.launch_instance()
'''


class NotebookKernel:
    """Keep variables between steps and return logs, figures, and failures."""

    def __init__(self, python_path, working_directory, environment):
        python_path = Path(python_path).absolute()
        bridge_root = Path(working_directory) / ("mangoleaf-bridge-" + uuid.uuid4().hex)
        specification_root = bridge_root / "kernels"
        specification = specification_root / "mangoleaf-v440"
        specification.mkdir(parents=True, exist_ok=False)
        profile_root = bridge_root / "ipython"
        profile_root.mkdir()
        launcher = bridge_root / "start-kernel.py"
        launcher.write_text(notebook_kernel_launcher(), encoding="utf-8")
        self.startup_log = bridge_root / "kernel-startup.log"
        kernel_environment = dict(environment)
        kernel_environment["IPYTHONDIR"] = str(profile_root)
        (specification / "kernel.json").write_text(json.dumps({
            "argv": [
                str(python_path), "-I", str(launcher), "-f", "{connection_file}",
                "--IPKernelApp.ipython_dir=" + str(profile_root),
                "--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel",
            ],
            "display_name": "MangoLeaf isolated Python", "language": "python",
        }), encoding="utf-8")

        # Connection keys stay in local runtime storage, never in Drive evidence.
        self.manager = KernelManager(
            kernel_name="mangoleaf-v440",
            kernel_spec_manager=KernelSpecManager(kernel_dirs=[str(specification_root)]),
            connection_file=str(bridge_root / "connection.json"),
            ip="127.0.0.1",
            autorestart=False,
        )
        self.client = None
        self.ready = False
        self.running = False
        try:
            # Keep process startup output locally, apart from Drive evidence
            # and the connection file containing the authentication key.
            with self.startup_log.open("w", encoding="utf-8") as startup_output:
                self.manager.start_kernel(
                    cwd=str(working_directory), env=kernel_environment,
                    stdout=startup_output, stderr=startup_output,
                )
                self.client = self.manager.blocking_client()
                self.client.start_channels()
                self.client.wait_for_ready(timeout=120)
                self.ready = True
        except Exception as error:
            self.close()
            raise RuntimeError(
                "The isolated kernel could not start. Inspect its local startup log: "
                + str(self.startup_log)
            ) from error
        except BaseException:
            self.close()
            raise

    def close(self):
        """Stop the process so interrupted fitting cannot continue unseen."""
        self.ready = False
        try:
            if self.manager.has_kernel:
                self.manager.shutdown_kernel(now=True)
        finally:
            if self.client is not None:
                self.client.stop_channels()

    def execute(self, source):
        if not self.ready:
            raise RuntimeError("The isolated kernel is unavailable. Repeat Step 1, then restore the appropriate saved run.")
        if self.running:
            raise RuntimeError("An isolated cell is still running. Wait for its result before starting another step.")
        self.running = True
        failure = None
        try:
            request_id = self.client.execute(source, allow_stdin=False, stop_on_error=True)
            while True:
                try:
                    message = self.client.get_iopub_msg(timeout=1)
                except Empty:
                    if not self.manager.is_alive():
                        self.close()
                        raise RuntimeError("The isolated Python process stopped. Repeat Step 1 and use the documented recovery controls.")
                    continue
                # Ignore output from a previous request or kernel startup.
                if message.get("parent_header", {}).get("msg_id") != request_id:
                    continue
                kind = message["msg_type"]
                content = message["content"]
                if kind == "status" and content["execution_state"] == "idle":
                    break
                if kind == "stream":
                    target = sys.stderr if content["name"] == "stderr" else sys.stdout
                    target.write(content["text"])
                    target.flush()
                elif kind in ("display_data", "execute_result", "update_display_data"):
                    if kind == "update_display_data":
                        get_ipython().display_pub.publish(data=content["data"], metadata=content.get("metadata", {}), transient=content.get("transient", {}), update=True)
                    else:
                        display(content["data"], raw=True, metadata=content.get("metadata", {}), transient=content.get("transient", {}))
                elif kind == "clear_output":
                    clear_output(wait=content.get("wait", False))
                elif kind == "error":
                    failure = content
                    print("\n".join(content.get("traceback", [])), file=sys.stderr, flush=True)

            # The shell reply is checked as well, so an execution error cannot
            # turn into a successful green cell after its traceback is printed.
            while True:
                reply = self.client.get_shell_msg(timeout=30)
                if reply.get("parent_header", {}).get("msg_id") == request_id:
                    break
            if reply["content"].get("status") != "ok":
                failure = failure or reply["content"]
            if failure:
                raise RuntimeError("Isolated cell stopped: " + failure.get("ename", "execution error") + ": " + failure.get("evalue", "Inspect the traceback above."))
        except KeyboardInterrupt:
            self.close()
            raise RuntimeError("Execution interrupted; the isolated process was stopped. Repeat Step 1 and follow saved-run recovery before continuing.") from None
        except Empty:
            self.close()
            raise RuntimeError("The isolated kernel did not complete its reply. Repeat Step 1 before continuing.") from None
        finally:
            self.running = False


def start_notebook_kernel(review):
    """Register the explicit marker used by every scientific notebook cell."""
    previous = globals().get("_mangoleaf_runner")
    if previous is not None:
        previous.close()
    runner = NotebookKernel(review["trainingPython"], Path("/content"), review["trainingEnvironment"])
    globals()["_mangoleaf_runner"] = runner
    atexit.register(runner.close)

    def mangoleaf_cell(line, source):
        if line.strip():
            raise ValueError("The mangoleaf cell marker takes no arguments.")
        runner.execute(source)

    get_ipython().register_magic_function(mangoleaf_cell, magic_kind="cell", magic_name="mangoleaf")
    # Send only explicit paths and evidence, never a complete host environment.
    worker_record = {key: str(review[key]) if review[key] is not None else None for key in ("evidenceDirectory", "trainingPython", "converterPython", "converterCommand")}
    worker_record["record"] = review["record"]
    initialization = "import json, os, sys\nfrom pathlib import Path\nRUNTIME_REVIEW = json.loads(" + repr(json.dumps(worker_record)) + ")\n"
    initialization += "assert sys.prefix != sys.base_prefix, 'Expected isolated Python'\nassert str(Path(sys.executable).absolute()) == RUNTIME_REVIEW['trainingPython'], 'Wrong Python interpreter'\n"
    initialization += "RUNTIME_REVIEW['evidenceDirectory'] = Path(RUNTIME_REVIEW['evidenceDirectory'])\n"
    initialization += "RUNTIME_REVIEW['converterEnvironment'] = dict(os.environ)\nRUNTIME_REVIEW['converterEnvironment'].update(TF_USE_LEGACY_KERAS='1', CUDA_VISIBLE_DEVICES='-1')\n"
    initialization += "get_ipython().run_line_magic('matplotlib', 'inline')\nprint('Scientific cells use:', sys.executable)\n"
    try:
        runner.execute(initialization)
    except BaseException:
        runner.close()
        raise
    return runner

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Host packages stay untouched. Step 1 can take several minutes on a new session.
RUNTIME_REVIEW = prepare_reviewed_runtime(
    Path("/content/drive/MyDrive/mangoleaf_experiments/v4.4.0/runtime_preflight")
)
start_notebook_kernel(RUNTIME_REVIEW)

In [ ]:
%%mangoleaf
import csv
import hashlib
import importlib.metadata as package_metadata
import json
import math
import os
import random
import shutil
import subprocess
import sys
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import scipy
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.utils.class_weight import compute_class_weight

CONVERTER_PYTHON = str(RUNTIME_REVIEW["converterPython"])
CONVERTER_COMMAND = str(RUNTIME_REVIEW["converterCommand"])
CONVERTER_ENVIRONMENT = RUNTIME_REVIEW["converterEnvironment"]

distributions = {}
for name in ("tensorflow", "keras", "Pillow", "ImageHash", "numpy", "scipy", "matplotlib", "scikit-learn"):
    distributions[name] = package_metadata.version(name)

RUNTIME = {
    "python": sys.version,
    "platform": sys.platform,
    "distributions": distributions,
    "runtimeReview": RUNTIME_REVIEW["record"],
}
print(json.dumps(RUNTIME, indent=2))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Select a Colab GPU runtime before beginning this workflow.")

tf.keras.mixed_precision.set_global_policy("float32")

## Step 2: Choose input folders and the output location

**What:** Set three independent raw input locations and the new output root.

**Why:** Exact input paths and settings must be recorded before a new run starts.

**How:** The mango input may be a folder containing the seven class folders, a parent of that folder, or the original ZIP. Background and OOD folders are read only. The 200 OOD images are inventoried here; their separate tuning/report assignment occurs in the threshold notebook.

**Expected result:** Printed paths point to the uploaded collections. Counts are checked later. Output directories never overlap source folders.

In [ ]:
%%mangoleaf
DRIVE_ROOT = Path("/content/drive/MyDrive")
RAW_MANGO_INPUT = DRIVE_ROOT / "final_mangoleafai_dataset_public_and_bpi"
BACKGROUND_INPUT = DRIVE_ROOT / "mangoleaf_background_photos_v4_4_0"
OOD_INPUT = DRIVE_ROOT / "mango_ood_test_v4_4_0"
OUTPUT_ROOT = DRIVE_ROOT / "mangoleaf_experiments/v4.4.0/training"

EXPECTED_RAW_IMAGE_COUNT = 5098
EXPECTED_BACKGROUND_COUNT = 225
EXPECTED_OOD_COUNT = 200

SEED = 42
CLASS_NAMES = [
    "anthracnose", "bacterial_black_spot", "bacterial_canker",
    "healthy_fruit", "healthy_leaf", "mango_scab", "sooty_mold",
]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
PHASH_DISTANCE = 5
MODEL_VERSION = "model-v4.4.0-candidate"
EXPERIMENT_VERSION = "v4.4.0"

STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 10
STAGE1_LEARNING_RATE = 0.001
STAGE2_LEARNING_RATE = 0.00001

BACKGROUND_PROBABILITY = 0.30
LOCAL_LIGHT_PROBABILITY = 0.30
LIGHT_AMPLITUDE = (18.0, 40.0)
LIGHT_RADIUS = (0.18, 0.35)

# Ordinary runs always create a new directory. Export recovery never calls model.fit.
RUN_MODE = "new"  # Allowed values: "new" or "export_only".
RECOVER_TRAINING_RUN = ""  # Exact v4.4.0 run directory, used only for export recovery.

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

for path in (RAW_MANGO_INPUT, BACKGROUND_INPUT, OOD_INPUT, OUTPUT_ROOT):
    path.resolve().relative_to(DRIVE_ROOT.resolve())
for input_path in (RAW_MANGO_INPUT, BACKGROUND_INPUT, OOD_INPUT):
    if not input_path.exists():
        raise FileNotFoundError(f"Input does not exist: {input_path}. Correct this setting before continuing.")
    if OUTPUT_ROOT.resolve() == input_path.resolve() or OUTPUT_ROOT.resolve().is_relative_to(input_path.resolve()) or input_path.resolve().is_relative_to(OUTPUT_ROOT.resolve()):
        raise ValueError("Input and output locations must be separate.")

print("Mango input:", RAW_MANGO_INPUT)
print("Background input:", BACKGROUND_INPUT)
print("OOD input:", OOD_INPUT)
print("New output root:", OUTPUT_ROOT)

## Step 3: Create safe file helpers and a unique run

**What:** Create reusable file, image, and fingerprint functions.

**Why:** A fingerprint identifies content even when names change. Safe writes prevent accidentally replacing completed evidence.

**How:** The notebook records both file bytes and decoded RGB pixels. Perceptual hashes identify possible visual similarities for review. The model loader keeps RGB values in the 0–255 range and lets MobileNet handle its own scaling.

**Expected result:** A unique run directory and runtime record are saved. Export-only recovery verifies the frozen inputs and existing fitted checkpoint instead of restarting training.

In [ ]:
%%mangoleaf
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
PREPROCESSING = "TensorFlow RGB decode; Pillow EXIF RGB for WebP; bilinear 224x224 without antialias; float32 0-255; MobileNet rescales internally"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def object_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_json_once(path, value):
    path = Path(path)
    # An identical rerun is harmless. A changed record requires a new run.
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError(f"Refusing to overwrite frozen evidence: {path}")
        return
    with path.open("x", encoding="utf-8", newline="\n") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False)
        handle.write("\n")


def safe_child(root, relative_path):
    if not isinstance(relative_path, str) or not relative_path or relative_path == "." or "\\" in relative_path or ":" in relative_path or "\x00" in relative_path:
        raise ValueError(f"Unsafe relative path: {relative_path}")
    value = PurePosixPath(relative_path)
    if value.is_absolute() or ".." in value.parts or not value.parts:
        raise ValueError(f"Unsafe relative path: {relative_path}")
    destination = Path(root).joinpath(*value.parts).resolve()
    destination.relative_to(Path(root).resolve())
    return destination


def raw_rgb(path):
    with Image.open(path) as opened:
        return ImageOps.exif_transpose(opened).convert("RGB")


def has_webp_content(path):
    """Read the file signature because a WebP photo may have a .jpg name."""
    with Path(path).open("rb") as source:
        header = source.read(12)
    return header[:4] == b"RIFF" and header[8:12] == b"WEBP"


def load_pixels(path):
    path = Path(path)
    if has_webp_content(path):
        image = tf.convert_to_tensor(np.asarray(raw_rgb(path)), dtype=tf.uint8)
    else:
        image = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
    return tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method="bilinear", antialias=False)


def predict_records(model, rows):
    """Predict unchanged images in manifest order, including during export recovery."""
    def examples():
        for row in rows:
            yield load_pixels(row["path"]).numpy().astype(np.float32)
    dataset = tf.data.Dataset.from_generator(
        examples, output_signature=tf.TensorSpec((224, 224, 3), tf.float32),
    ).batch(BATCH_SIZE).prefetch(1)
    return model.predict(dataset, verbose=0)


def image_inventory(folder, pool, labelled=False):
    rows = []
    failures = []
    for path in sorted(Path(folder).rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        relative = path.relative_to(folder).as_posix()
        try:
            pixels = raw_rgb(path)
            decoded_hash = hashlib.sha256(str(pixels.size).encode("utf-8") + pixels.tobytes()).hexdigest()
            # Also verify the actual inference loader, not only Pillow's decoder.
            loaded = load_pixels(path).numpy()
            if loaded.shape != (224, 224, 3) or not np.isfinite(loaded).all():
                raise ValueError("Invalid model input pixels")
            row = {
                "id": file_sha256(path)[:20], "path": str(path), "pool": pool,
                "sourceRelativePath": relative, "sha256": file_sha256(path),
                "bytes": path.stat().st_size, "decodedRgbSha256": decoded_hash,
                "phash": str(imagehash.phash(pixels, hash_size=8)),
                "dimensions": list(pixels.size), "readable": True,
            }
            if labelled:
                row["label"] = PurePosixPath(relative).parts[0]
                if row["label"] not in CLASS_NAMES:
                    raise ValueError(f"Unexpected class folder: {row['label']}")
            rows.append(row)
        except Exception as error:
            failures.append({"relativePath": relative, "error": str(error)})
    if failures:
        raise RuntimeError("Unreadable or unsupported images require correction: " + json.dumps(failures[:20]))
    return rows


def portable_rows(rows):
    return [{key: value for key, value in row.items() if key != "path"} for row in rows]


def verify_image_rows(rows):
    for row in rows:
        if file_sha256(row["path"]) != row["sha256"]:
            raise RuntimeError("Input changed after inventory: " + row["path"])


def save_contact_sheets(rows, destination, title, columns=4):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for page_number, start in enumerate(range(0, len(rows), 20)):
        chosen = rows[start:start + 20]
        page = Image.new("RGB", (columns * 260, math.ceil(len(chosen) / columns) * 290 + 36), "white")
        drawing = ImageDraw.Draw(page)
        drawing.text((8, 8), title, fill="black")
        for index, row in enumerate(chosen):
            image = raw_rgb(row["path"])
            image.thumbnail((250, 240))
            left = (index % columns) * 260
            top = (index // columns) * 290 + 36
            page.paste(image, (left + 5, top))
            drawing.text((left + 5, top + 242), row["id"] + "\n" + row["sourceRelativePath"][:34], fill="black")
        page.save(destination / f"page_{page_number:03d}.png")


if RUN_MODE == "new":
    run_id = "v4-4-0-training-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    RUN_DIR = OUTPUT_ROOT / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    DATASET_DIR = RUN_DIR / "dataset"
    save_json_once(RUN_DIR / "runtime.json", RUNTIME)
    shutil.copytree(RUNTIME_REVIEW["evidenceDirectory"], RUN_DIR / "runtime_preflight")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
    (RUN_DIR / "requirements_snapshot.txt").write_text(freeze, encoding="utf-8")
elif RUN_MODE == "export_only":
    RUN_DIR = Path(RECOVER_TRAINING_RUN).resolve()
    RUN_DIR.relative_to(OUTPUT_ROOT.resolve())
    if (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("This run is complete. Use its saved artifacts; do not overwrite them.")
    fitted_state = read_json(RUN_DIR / "training_state.json")
    fitted_runtime = read_json(RUN_DIR / "runtime.json")
    if any(fitted_runtime["distributions"][name] != RUNTIME["distributions"][name] for name in ("tensorflow", "keras")):
        raise RuntimeError("Export recovery must use the fitted run's recorded TensorFlow and Keras versions.")
    for relative, expected in fitted_state["artifacts"].items():
        if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
            raise RuntimeError("Recovery artifact differs from its saved fingerprint: " + relative)
    DATASET_DIR = RUN_DIR / "dataset"
    manifest = read_json(RUN_DIR / "input_manifest.json")
    records = [dict(row, path=str(safe_child(DATASET_DIR, row["relativePath"]))) for row in manifest["images"]]
    verify_image_rows(records)
    plan = read_json(RUN_DIR / "plan.json")
    if plan["classNames"] != CLASS_NAMES or plan["experimentVersion"] != EXPERIMENT_VERSION:
        raise RuntimeError("Recovery must use this notebook's own v4.4.0 class order.")
    train_rows = [row for row in records if row["subset"] == "train"]
    val_rows = [row for row in records if row["subset"] == "val"]
    selection = read_json(RUN_DIR / "example_selection.json")
    by_id = {row["id"]: row for row in records}
    example_rows = [by_id[item["id"]] for item in selection["images"]]
    training_history = read_json(RUN_DIR / "training_history.json")
    selected_model = tf.keras.models.load_model(RUN_DIR / "model_selected.keras", compile=False, safe_mode=True)
    scene_selection = set(read_json(RUN_DIR / "scene_review.json")["approvedIds"])
    approved_scenes = [dict(row, path=str(safe_child(BACKGROUND_INPUT, row["sourceRelativePath"])))
                       for row in read_json(RUN_DIR / "scene_inventory.json")["images"] if row["id"] in scene_selection]
    if {row["id"] for row in approved_scenes} != scene_selection:
        raise RuntimeError("Recorded approved scene identities are missing during export recovery.")
    verify_image_rows(approved_scenes)
    print("Recovery mode: skip Steps 4 through 14; continue with calibration and export.")
else:
    raise ValueError("RUN_MODE must be new or export_only.")

print("Preserve this exact run directory:", RUN_DIR)

## Step 4: Inventory every raw photo

**What:** Read the raw mango, background, and OOD collections without changing them.

**Why:** Actual image counts and fingerprints resolve upload-count differences and prevent unrecorded source changes.

**How:** If the mango input is a ZIP, safely extract it under Colab's temporary working directory. Locate exactly one folder containing all seven class folders. Detect WebP from its file bytes, even when the name ends in .jpg; use the existing Pillow WebP loader. Native TensorFlow formats keep their decoder. Save full inventories, including images that may later be excluded from training.

**Expected result:** Expect 5,098 raw mango images, 225 backgrounds, and 200 OOD images. Changed counts or unreadable input stop for review; no source image is rewritten and no old split is loaded.

In [ ]:
%%mangoleaf
if RUN_MODE != "new":
    raise RuntimeError("Skip raw inventory during export-only recovery.")

if RAW_MANGO_INPUT.is_file():
    if RAW_MANGO_INPUT.suffix.lower() != ".zip":
        raise ValueError("The raw mango file must be a ZIP.")
    extraction_root = Path("/content/mangoleaf-v4-4-0-inputs") / RUN_DIR.name
    extraction_root.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(RAW_MANGO_INPUT) as archive:
        for item in archive.infolist():
            destination = safe_child(extraction_root, item.filename)
            if item.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, destination.open("xb") as output:
                    shutil.copyfileobj(source, output)
    mango_search_root = extraction_root
else:
    mango_search_root = RAW_MANGO_INPUT

possible_roots = [mango_search_root] + [path for path in mango_search_root.rglob("*") if path.is_dir()]
class_roots = [path for path in possible_roots if all((path / label).is_dir() for label in CLASS_NAMES)]
if len(class_roots) != 1:
    raise ValueError("Choose the one raw mango folder containing the seven class folders. Found: " + str(class_roots))
RAW_MANGO_ROOT = class_roots[0]

raw_rows = image_inventory(RAW_MANGO_ROOT, "mango", labelled=True)
background_rows = image_inventory(BACKGROUND_INPUT, "background")
ood_rows = image_inventory(OOD_INPUT, "ood")

actual_counts = {"mango": len(raw_rows), "background": len(background_rows), "ood": len(ood_rows)}
expected_counts = {"mango": EXPECTED_RAW_IMAGE_COUNT, "background": EXPECTED_BACKGROUND_COUNT, "ood": EXPECTED_OOD_COUNT}
print("Actual readable image counts:", actual_counts)
print("Raw class counts:", dict(Counter(row["label"] for row in raw_rows)))
if actual_counts != expected_counts:
    raise RuntimeError("Image counts differ. Check the actual folder contents and explicitly correct expected counts before creating a new run.")

save_json_once(RUN_DIR / "mango_inventory.json", {"images": portable_rows(raw_rows), "count": len(raw_rows)})
save_json_once(RUN_DIR / "scene_inventory.json", {"images": portable_rows(background_rows), "count": len(background_rows)})
save_json_once(RUN_DIR / "ood_inventory.json", {"images": portable_rows(ood_rows), "count": len(ood_rows), "previouslyUsedStudyPhotos": True})

# An exact overlap between task images and external pools would contaminate evaluation.
external_overlaps = []
for field in ("sha256", "decodedRgbSha256"):
    mango_values = {row[field] for row in raw_rows}
    scene_values = {row[field] for row in background_rows}
    for row in ood_rows:
        if row[field] in mango_values or row[field] in scene_values:
            external_overlaps.append({"field": field, "oodId": row["id"], "sourceRelativePath": row["sourceRelativePath"]})
    for row in background_rows:
        if row[field] in mango_values:
            external_overlaps.append({"field": field, "backgroundId": row["id"], "sourceRelativePath": row["sourceRelativePath"]})
save_json_once(RUN_DIR / "external_overlap_audit.json", {"exactOverlaps": external_overlaps})
if external_overlaps:
    raise RuntimeError("Exact overlap exists between input pools. Review it before creating a new run.")

## Step 5: Review duplicates and related photos

**What:** Find exact copies and flag visually similar images.

**Why:** Copies or related captures must not appear in both training and evaluation. The 62 previously reviewed distinct images remain visible and retained unless an actual exact-copy conflict is resolved explicitly.

**How:** Keep one representative per identical decoded image, favoring a named example or known reviewed identity. Perceptual similarities are grouped conservatively, not deleted. Inspect pair sheets and the audit, then record any reviewed distinct pairs and additional related-photo groups.

**Expected result:** A fresh, auditable retained list and grouping policy are approved. These literal review identities are evidence only; no earlier run folder is needed.

In [ ]:
%%mangoleaf
# These 62 literal fingerprints preserve the reviewed photo identities, not an old split.
REVIEWED_IMAGES = [
  {
    "label": "anthracnose",
    "currentName": "verified mango leaf disease 21.jpg",
    "historicalName": "verified mango leaf disease 21.jpg",
    "sha256": "a557157b1e887d6f8f7067efbaa97447074114b2967c3c2337656d1936b45102"
  },
  {
    "label": "anthracnose",
    "currentName": "verified mango leaf disease 20.jpg",
    "historicalName": "verified mango leaf disease 20.jpg",
    "sha256": "87dad59aa6422d959e3fcfffd35ddca9c918e858dd7b220f1e016ae09d741f6f"
  },
  {
    "label": "anthracnose",
    "currentName": "DSC_0119.JPG",
    "historicalName": "DSC_0119.JPG",
    "sha256": "01e6fd123c9ae054d2d904fef25f543279c6c74914d44a33cd9c7488ca2861c6"
  },
  {
    "label": "anthracnose",
    "currentName": "DSC_0488.JPG",
    "historicalName": "DSC_0488.JPG",
    "sha256": "ee9676472204c9cdd0830dce3be63a9d47e646ecb9d2c70126ddcdac063e4ecd"
  },
  {
    "label": "anthracnose",
    "currentName": "DSC_0970.JPG",
    "historicalName": "DSC_0970.JPG",
    "sha256": "8b5d0b0e2500ff762a773b326c6a0a3ab545e51e682a253d3776be738d61123c"
  },
  {
    "label": "anthracnose",
    "currentName": "Anthracnose on young leaves.jpeg",
    "historicalName": "Anthracnose on young leaves.jpeg",
    "sha256": "5c10b4851fd67db6b5a21677e725d124491633afea45186fbe2e2a6f9bd554ec"
  },
  {
    "label": "anthracnose",
    "currentName": "AnthracnoseonMangoLeaf.jpeg",
    "historicalName": "AnthracnoseonMangoLeaf.jpeg",
    "sha256": "52a0fa64a504139aa65e6cbf807ecae21be6f4933bc4c1d5e2d8797a54de7227"
  },
  {
    "label": "sooty_mold",
    "currentName": "IMG_20260219_135000.jpg",
    "historicalName": "IMG_20260219_135000.jpg",
    "sha256": "2a22ee7f89ea03ad9ee924b5815ae71576d6ccd3305576dccc6f3bf00e31fce3"
  },
  {
    "label": "sooty_mold",
    "currentName": "viber_image_2024-11-15_13-17-51-567.jpg",
    "historicalName": "viber_image_2024-11-15_13-17-51-567.jpg",
    "sha256": "d785e72ca8291a194d5f56f2c75d2087ca3fd9728549246b888a3b964ce5b3c9"
  },
  {
    "label": "sooty_mold",
    "currentName": "IMG_20260219_134935.jpg",
    "historicalName": "IMG_20260219_134935.jpg",
    "sha256": "6c1c6c90f2dbfdb3ab1d451b90a9ce2769222aaa237bb820bf55f782e63498d5"
  },
  {
    "label": "sooty_mold",
    "currentName": "DSC_1029.JPG",
    "historicalName": "DSC_1029.JPG",
    "sha256": "7dfd201d72ea78a4557fdc5c5bddb594af538ff0270e020e73c9f382b3fa6f74"
  },
  {
    "label": "sooty_mold",
    "currentName": "724869371_27534362222843583_7670640125862653147_n.jpg",
    "historicalName": "724869371_27534362222843583_7670640125862653147_n.jpg",
    "sha256": "81c5a02299de1f2d66627a0f709a4f0c3235066749f7f8b203840b86f8a93b3c"
  },
  {
    "label": "sooty_mold",
    "currentName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
    "historicalName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
    "sha256": "32d90fe257d7ef563ef8166768b638a92a95c8196d33de90ced349c22b62a410"
  },
  {
    "label": "healthy_leaf",
    "currentName": "HealthyLeaf_2.jpg",
    "historicalName": "HealthyLeaf_2.jpg",
    "sha256": "e2e68e23f0fd8f88b999c0740cb577e126e78de61fa1fe71772d908ea2fc6375"
  },
  {
    "label": "healthy_leaf",
    "currentName": "HealthyLeaf_1.jpg",
    "historicalName": "HealthyLeaf_1.jpg",
    "sha256": "90a7907f7a259e404f6bb8c6558bb1081f2d2942249ebb5cac4b747a2f989e0c"
  },
  {
    "label": "mango_scab",
    "currentName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
    "historicalName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
    "sha256": "5a10c88561c498889b7668319c36fa1a432b942c21832e65d8c4a3063dbcf78b"
  },
  {
    "label": "mango_scab",
    "currentName": "DSC_0281.JPG",
    "historicalName": "DSC_0281.JPG",
    "sha256": "da1e429fc670c52fd75ba8cfe37b019606841ee7e012182f5696895e1ba13132"
  },
  {
    "label": "mango_scab",
    "currentName": "DSC_0966.JPG",
    "historicalName": "DSC_0966.JPG",
    "sha256": "de8cd5a96aefcccbd838a67411d44f13dcf9bf647200e40c71b7da28f26a1724"
  },
  {
    "label": "mango_scab",
    "currentName": "Scab.JPG",
    "historicalName": "Scab.JPG",
    "sha256": "10344e2e95ceeafab7d77e3e28e4ebbdebd699ec7e862c9c011516038e821068"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 70.jpg",
    "historicalName": "non verified mango disease 70.jpg",
    "sha256": "47a8192ddf400d3ba6397ba7d3acb21142d75267b80abe3911ba0d76e85ce995"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 71.jpg",
    "historicalName": "non verified mango disease 71.jpg",
    "sha256": "0c46b7f3c43190fb6e5663d6248f81e1eca31287ea8ab8cf3c2c9113aa46a848"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 73.jpg",
    "historicalName": "non verified mango disease 73.jpg",
    "sha256": "7562eee067e4a968c8f7c1f3281125bcb3546dcab2795f196ee11286e836ac09"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 68.jpg",
    "historicalName": "non verified mango disease 68.jpg",
    "sha256": "281064014381e3f9d4cd163b62ec6e643364ee5a3a0840875f9c310912a30c8d"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 67.jpg",
    "historicalName": "non verified mango disease 67.jpg",
    "sha256": "6e99067feab48b78d0e0c4260e342c1b64673dab00a5c18099310bb9053f1dd7"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 66.jpg",
    "historicalName": "non verified mango disease 66.jpg",
    "sha256": "0e515243d9cec0d0e1f970d4ba79081606c2280ee02176e30fa24b2817784cda"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 65.jpg",
    "historicalName": "non verified mango disease 65.jpg",
    "sha256": "ae4bfcc5cd3eba0f652aa4355d1780ef6efe27af71f3d020a15b1eee2c0f975b"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 56.jpg",
    "historicalName": "non verified mango disease 56.jpg",
    "sha256": "ad4415243dc86b38c7ac7f529fdb2e09f7fc9410ded477a553a0f33ef02736fa"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 53.jpg",
    "historicalName": "non verified mango disease 53.jpg",
    "sha256": "89652a16e8c0590a4fd51b9ff66c7ff9fb95a7ded8aece63c85565dad39f5fff"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 52.jpg",
    "historicalName": "non verified mango disease 52.jpg",
    "sha256": "cfb52caae3133082bc5f70c82f4b4c73589a65087409383e87453d405648391e"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 51.jpg",
    "historicalName": "non verified mango disease 51.jpg",
    "sha256": "e2c7e67b3026efe9eac0941272a41654f54c8317a2d11d7ba0eb81699891a2bd"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 49.jpg",
    "historicalName": "non verified mango disease 49.jpg",
    "sha256": "7b87cb61eb26ffffde585b8cbb254795f0da834ce1c306bae76fe2b6cdf0f736"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 11.jpg",
    "historicalName": "non verified mango disease 11.jpg",
    "sha256": "81a2f9abde6762ed891db43c570a5339d465682e3817c4f79917f8f497ed77cd"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 10.jpg",
    "historicalName": "verified mango disease 10.jpg",
    "sha256": "afcb278a84a3abf145a2d396bbcfa99b5818b217764c3fef07246d6993421095"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 9.jpg",
    "historicalName": "verified mango disease 9.jpg",
    "sha256": "6fdc1ff1319c85198ae2d42978e9c3ec11ee82b754e982915a3df896e922e187"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 1.jpg",
    "historicalName": "verified mango disease 1.jpg",
    "sha256": "156d089cee4c22e7256a0aa6ef48c84025507fcafda3b463fe39f9d9f852cd87"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 58.jpg",
    "historicalName": "verified mango disease 58.jpg",
    "sha256": "7fe031c22293d0c980ddf2744d2b3732bcbe22c659281c67ef7f24ce577619c6"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 36.jpg",
    "historicalName": "verified mango disease 36.jpg",
    "sha256": "22deb301d133db0af636c788508379203b6b90e4ca3ce9aa3ce238069cccfc4e"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 32.jpg",
    "historicalName": "verified mango disease 32.jpg",
    "sha256": "bbfbb925ba9bc8cb2595d501ab8180bc7378dfb199a1110793743e79a40e376f"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 6.jpg",
    "historicalName": "verified mango disease 6.jpg",
    "sha256": "262ea4d6d82594a4333470180b3de1a77560fafc3acc1e0a75144260e1928e15"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 2.jpg",
    "historicalName": "verified mango disease 2.jpg",
    "sha256": "1da80fe4b68725c7c39d78af8e4a42c65316c4e29d2be3756bef74c02a51d1d0"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 63.jpg",
    "historicalName": "verified mango disease 63.jpg",
    "sha256": "95e7184c9c0c2c0f5e729d16513410470c77e07bc4a787d50c06f79a55977cd8"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 57.jpg",
    "historicalName": "verified mango disease 57.jpg",
    "sha256": "9c9bc581355a1a4bfcd0befe2051135b7903d219df9320587cd88a28c94db5f6"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 7.jpg",
    "historicalName": "verified mango disease 7.jpg",
    "sha256": "0a7326dd87f77caad7cae79d709bd7cdfcb5a0aec2ef96463782b2920b5cb55c"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 55.jpg",
    "historicalName": "verified mango disease 55.jpg",
    "sha256": "d70902b669201181ee27847a13e4bbf684a371c18957b2cbb20b6cfcbf75275a"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 45.jpg",
    "historicalName": "verified mango disease 45.jpg",
    "sha256": "0827d4061caffa787cb8fab501fb3f25f6e4ef40f434ee03787136b0c8f2d4b7"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 54.jpg",
    "historicalName": "verified mango disease 54.jpg",
    "sha256": "4eee0b91cfa3f2c5b66a0f334d2d2e487fd666d7aa10b3ea9f8bdfc9d36dbfea"
  },
  {
    "label": "mango_scab",
    "currentName": "verified mango disease 8.jpg",
    "historicalName": "verified mango disease 8.jpg",
    "sha256": "00239798d21836b06afea3d213aff445d509eabab9cfa5515e687afc20447e97"
  },
  {
    "label": "mango_scab",
    "currentName": "DSC_1014.JPG",
    "historicalName": "DSC_1014.JPG",
    "sha256": "348841118bd2fed6966a5837ddd7dcde55cb053057371a39ce4f051aa5627bdf"
  },
  {
    "label": "mango_scab",
    "currentName": "Scab1.JPG",
    "historicalName": "Scab1.JPG",
    "sha256": "dbf4ee9f20018da3f04c542d9b369403bb8fbf0149b36c14a4363782d68d174e"
  },
  {
    "label": "mango_scab",
    "currentName": "725146731_2449764938863852_7559841505625140608_n.jpg",
    "historicalName": "725146731_2449764938863852_7559841505625140608_n.jpg",
    "sha256": "a86e5a775f479a63fe2528305bc350300435f8b048494a06ae13e1fd78d0d0f5"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_1021.JPG",
    "historicalName": "DSC_1021.JPG",
    "sha256": "9872562f0d3dfe137e0db5edfbaeaa602aef7444aacded8d18ce9de1bfbf5390"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_1019.JPG",
    "historicalName": "DSC_1019.JPG",
    "sha256": "f744b6410bd2ba2897372201bd1179144fbcb6860e11cf2ba083e1fc8e4f1d9e"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_0436.JPG",
    "historicalName": "DSC_0436.JPG",
    "sha256": "7053f37a4c34867f2c77d44ca764def21315667a24ce4cf388586a6be866d458"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_0205.JPG",
    "historicalName": "DSC_0205.JPG",
    "sha256": "0a1ab9392878b940fc828aa7e7659b0ee2897244935f0dd57e8e012384c9cc53"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_0204.JPG",
    "historicalName": "DSC_0204.JPG",
    "sha256": "5e832bc5a93f67dda12f79405fa39e89f7208052bb5ee6e627711967c6ee668c"
  },
  {
    "label": "healthy_fruit",
    "currentName": "DSC_0201 (1).JPG",
    "historicalName": "DSC_0201 (1).JPG",
    "sha256": "84e0237792a86d60a92e12a4da910e7bfe20f60439aff39f3534d6eaeb1c35f4"
  },
  {
    "label": "healthy_fruit",
    "currentName": "727799974_830567853245491_1446073817623999359_n.jpg",
    "historicalName": "727799974_830567853245491_1446073817623999359_n.jpg",
    "sha256": "9e0436c4d16f0517ffb395477f0567ff046fd41d04311ae74c1abb224c496504"
  },
  {
    "label": "healthy_fruit",
    "currentName": "725318273_1696519271391031_8474708639771176175_n.jpg",
    "historicalName": "725318273_1696519271391031_8474708639771176175_n.jpg",
    "sha256": "18c047f48c6deb7b423919369488485c608300cbba79d078428954694073ba4f"
  },
  {
    "label": "healthy_fruit",
    "currentName": "723960758_2381115535716542_3913785565978577803_n.jpg",
    "historicalName": "723960758_2381115535716542_3913785565978577803_n.jpg",
    "sha256": "3fb605dbbc492f8466267cbb8ec52ceede4455c676dc54a66db37b7e909ed644"
  },
  {
    "label": "healthy_fruit",
    "currentName": "723624552_976005712089787_680463173797746853_n.jpg",
    "historicalName": "723624552_976005712089787_680463173797746853_n.jpg",
    "sha256": "bccae50356a84611268b9fd0b5f5bf5517b222ac223d298084b121e169550b64"
  },
  {
    "label": "healthy_fruit",
    "currentName": "723624552_976005712089787_680463173797746853_n_2.jpg",
    "historicalName": "723624552_976005712089787_680463173797746853_n_2.jpg",
    "sha256": "b1c8ddc707402b023f61be78b900eb878af012c14964e1dbf25b7aab060c2310"
  },
  {
    "label": "healthy_fruit",
    "currentName": "719671069_1291039922786074_6951448894192280220_n.jpg",
    "historicalName": "719671069_1291039922786074_6951448894192280220_n.jpg",
    "sha256": "61cabb3193580b025925fe0177dfdf9a62cf7ea19a18df33218c41842349439c"
  }
]

In [ ]:
%%mangoleaf
GRADCAM_NAMES = {
    "anthracnose": "verified mango leaf disease 21.jpg",
    "sooty_mold": "DSC_1029.JPG",
    "healthy_leaf": "HealthyLeaf_2.jpg",
    "mango_scab": "DSC_0281.JPG",
    "healthy_fruit": "DSC_1021.JPG",
}
reviewed_hashes = {row["sha256"] for row in REVIEWED_IMAGES}
raw_by_hash = defaultdict(list)
decoded_groups = defaultdict(list)
for row in raw_rows:
    raw_by_hash[row["sha256"]].append(row)
    decoded_groups[row["decodedRgbSha256"]].append(row)

missing_reviewed = [row for row in REVIEWED_IMAGES if row["sha256"] not in raw_by_hash]
wrong_labels = [row for row in REVIEWED_IMAGES if any(raw["label"] != row["label"] for raw in raw_by_hash[row["sha256"]])]
if missing_reviewed or wrong_labels:
    raise RuntimeError("Known reviewed image identity/label is missing or changed. Inspect reviewed_images.json and raw inventory.")
save_json_once(RUN_DIR / "reviewed_images.json", REVIEWED_IMAGES)

retained_rows = []
exact_groups = []
for decoded_hash, group in sorted(decoded_groups.items()):
    if len({row["label"] for row in group}) != 1:
        raise RuntimeError("Identical decoded image has conflicting class labels: " + str([row["sourceRelativePath"] for row in group]))
    protected = [row for row in group if row["sha256"] in reviewed_hashes]
    if len({row["sha256"] for row in protected}) > 1:
        raise RuntimeError("Two reviewed distinct identities have identical decoded pixels. Resolve that conflict explicitly before a new run.")
    chosen = sorted(group, key=lambda row: (
        row["sha256"] not in reviewed_hashes,
        Path(row["sourceRelativePath"]).name != GRADCAM_NAMES.get(row["label"]),
        row["sourceRelativePath"],
    ))[0]
    retained_rows.append(chosen)
    if len(group) > 1:
        exact_groups.append({"keptId": chosen["id"], "keptPath": chosen["sourceRelativePath"], "copies": portable_rows(group)})

similarity_pairs = []
for index, first in enumerate(retained_rows):
    first_hash = int(first["phash"], 16)
    for second in retained_rows[index + 1:]:
        distance = (first_hash ^ int(second["phash"], 16)).bit_count()
        if distance <= PHASH_DISTANCE:
            similarity_pairs.append({"firstId": first["id"], "secondId": second["id"], "distance": distance,
                                     "firstLabel": first["label"], "secondLabel": second["label"]})

duplicate_audit = {
    "rawCount": len(raw_rows), "uniqueByteHashes": len(raw_by_hash),
    "retainedDecodedIdentities": len(retained_rows), "exactCopyGroups": exact_groups,
    "similarityDistance": PHASH_DISTANCE, "similarityPairs": similarity_pairs,
    "policy": "Remove only exact copies; retain visually distinct photos; group related captures before splitting.",
}
save_json_once(RUN_DIR / "duplicate_audit.json", duplicate_audit)
by_id = {row["id"]: row for row in retained_rows}
flagged_rows = [by_id[key] for key in sorted({pair[field] for pair in similarity_pairs for field in ("firstId", "secondId")})]
save_contact_sheets(flagged_rows, RUN_DIR / "duplicate_previews", "Possible similarities: inspect duplicate_audit.json pairs")
save_contact_sheets(retained_rows, RUN_DIR / "retained_previews", "Retained mango identities: review related captures")
print("Raw files:", len(raw_rows), "Retained exact identities:", len(retained_rows), "Possible similar pairs:", len(similarity_pairs))
print("Inspect duplicate_audit.json and all sheets in duplicate_previews/ and retained_previews/.")

In [ ]:
%%mangoleaf
# Edit these settings after reviewing the saved image sheets and acquisition notes.
DUPLICATE_REVIEW_COMPLETE = False
DISTINCT_SIMILAR_PAIRS = []  # Example: [["first-image-id", "second-image-id"]]. Both remain retained.
ADDITIONAL_RELATED_GROUPS = []  # Example: [["same-tree-photo-id-1", "same-tree-photo-id-2"]].
DUPLICATE_REVIEW_NOTES = ""
SOURCE_NOTES = {}  # Optional: image ID or source path mapped to manually entered origin/permission notes.

if not DUPLICATE_REVIEW_COMPLETE or not DUPLICATE_REVIEW_NOTES:
    raise RuntimeError("Review duplicate and related-photo sheets, enter notes, then approve this gate.")

parents = {row["id"]: row["id"] for row in retained_rows}


def group_root(key):
    while parents[key] != key:
        parents[key] = parents[parents[key]]
        key = parents[key]
    return key


def join_groups(first, second):
    left, right = group_root(first), group_root(second)
    parents[max(left, right)] = min(left, right)


distinct_pairs = {tuple(sorted(pair)) for pair in DISTINCT_SIMILAR_PAIRS}
valid_pairs = {tuple(sorted((pair["firstId"], pair["secondId"]))) for pair in similarity_pairs}
if not distinct_pairs.issubset(valid_pairs):
    raise ValueError("Distinct-pair decisions must refer to audited similarity pairs.")
for pair in similarity_pairs:
    if tuple(sorted((pair["firstId"], pair["secondId"]))) not in distinct_pairs:
        join_groups(pair["firstId"], pair["secondId"])
for group in ADDITIONAL_RELATED_GROUPS:
    if len(group) < 2 or any(key not in parents for key in group):
        raise ValueError("Every related group needs at least two known retained image IDs.")
    for key in group[1:]:
        join_groups(group[0], key)

for row in retained_rows:
    row["groupId"] = group_root(row["id"])
if not reviewed_hashes.issubset({row["sha256"] for row in retained_rows}):
    raise RuntimeError("A reviewed identity was removed. Stop and inspect the exact-copy audit.")
save_json_once(RUN_DIR / "duplicate_review.json", {
    "complete": True, "distinctSimilarPairs": DISTINCT_SIMILAR_PAIRS,
    "additionalRelatedGroups": ADDITIONAL_RELATED_GROUPS, "notes": DUPLICATE_REVIEW_NOTES,
    "sourceNotes": SOURCE_NOTES, "groupAssignments": {row["id"]: row["groupId"] for row in retained_rows},
})
print("Retained photographs:", len(retained_rows), "Independent grouping units:", len({row["groupId"] for row in retained_rows}))

## Step 6: Create and freeze the new 70/15/15 split

**What:** Assign whole photo groups to training, validation, or test.

**Why:** The same capture group must not cross subsets. Validation examples are reserved before splitting so Grad-CAM choices cannot be cherry-picked from test outcomes.

**How:** A deterministic grouped allocation targets 70%, 15%, and 15% per class. Reserve the five named identities and their groups in validation. Choose one validation example for each remaining class with a fixed random seed. Review actual counts before freezing the split.

**Expected result:** All seven classes appear in each subset, every group has one subset, and seven fixed example identities are saved. Actual proportions may differ because groups cannot be broken apart.

In [ ]:
%%mangoleaf
groups = defaultdict(list)
for row in retained_rows:
    groups[row["groupId"]].append(row)

named_examples = {}
for label, name in GRADCAM_NAMES.items():
    matches = [row for row in raw_rows if row["label"] == label and Path(row["sourceRelativePath"]).name == name]
    identities = {row["decodedRgbSha256"] for row in matches}
    if len(identities) != 1:
        raise ValueError("Named example is missing or ambiguous: " + name)
    named_examples[label] = next(row for row in retained_rows if row["decodedRgbSha256"] in identities)

forced_validation = {row["groupId"] for row in named_examples.values()}
subsets = ("train", "val", "test")
fractions = np.array([0.70, 0.15, 0.15])
class_totals = np.array([sum(row["label"] == label for row in retained_rows) for label in CLASS_NAMES], dtype=float)
targets = fractions[:, None] * class_totals[None, :]
counts = np.zeros_like(targets)
assignments = {}


def assign_mango_groups(groups, class_names, reserved_group_ids, seed):
    """Allocate complete related-photo groups while targeting class-balanced fractions."""
    subset_names = ("train", "val", "test")
    class_totals = np.array([sum(row["label"] == label for group in groups.values() for row in group)
                             for label in class_names], dtype=float)
    targets = np.array([0.70, 0.15, 0.15])[:, None] * class_totals[None, :]
    counts = np.zeros_like(targets)
    assignments = {}

    def contribution(group_id):
        return np.array([sum(row["label"] == label for row in groups[group_id]) for label in class_names], dtype=float)

    if not set(reserved_group_ids).issubset(groups):
        raise ValueError("Reserved validation groups must exist in the retained dataset.")
    for group_id in sorted(reserved_group_ids):
        assignments[group_id] = "val"
        counts[1] += contribution(group_id)

    unassigned = sorted(key for key in groups if key not in assignments)
    order = np.random.default_rng(seed).permutation(unassigned).tolist()
    order.sort(key=lambda key: -len(groups[key]))
    for group_id in order:
        values = contribution(group_id)
        scores = []
        for subset_index in range(3):
            proposed = counts.copy()
            proposed[subset_index] += values
            scores.append(float(np.sum(((proposed - targets) / np.maximum(targets, 1)) ** 2)))
        winner = int(np.argmin(scores))
        assignments[group_id] = subset_names[winner]
        counts[winner] += values

    # An impossible/oversized grouping requires review, never an automatic broken group.
    if np.any(counts == 0):
        raise RuntimeError("A class is absent from a subset. Review oversized related-photo groups; never split them across subsets.")
    return assignments, counts


assignments, counts = assign_mango_groups(groups, CLASS_NAMES, forced_validation, SEED)

records = []
for row in sorted(retained_rows, key=lambda item: (CLASS_NAMES.index(item["label"]), item["id"])):
    saved = dict(row)
    saved["subset"] = assignments[row["groupId"]]
    basename = Path(row["sourceRelativePath"]).name
    saved["relativePath"] = f"{saved['subset']}/{saved['label']}/{saved['id']}__{basename}"
    records.append(saved)

counts_by_subset = {subset: dict(Counter(row["label"] for row in records if row["subset"] == subset)) for subset in subsets}
print("Proposed class counts:", json.dumps(counts_by_subset, indent=2))
print("Actual subset fractions:", {subset: round(sum(row["subset"] == subset for row in records) / len(records), 4) for subset in subsets})

SPLIT_REVIEW_COMPLETE = False
SPLIT_REVIEW_NOTES = ""
if not SPLIT_REVIEW_COMPLETE or not SPLIT_REVIEW_NOTES:
    raise RuntimeError("Review the printed grouped split and reserved validation examples, then approve this gate.")

DATASET_DIR.mkdir(exist_ok=False)
verify_image_rows(raw_rows)
for row in records:
    destination = safe_child(DATASET_DIR, row["relativePath"])
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Path(row["path"]).open("rb") as source, destination.open("xb") as output:
        shutil.copyfileobj(source, output)
    row["path"] = str(destination)
verify_image_rows(records)

train_rows = [row for row in records if row["subset"] == "train"]
val_rows = [row for row in records if row["subset"] == "val"]
example_rows = []
example_rng = np.random.default_rng(SEED)
for label in CLASS_NAMES:
    if label in named_examples:
        selected_id = named_examples[label]["id"]
        selected = next(row for row in val_rows if row["id"] == selected_id)
        reason = "Named example reserved with its related-photo group before split freeze"
    else:
        choices = sorted((row for row in val_rows if row["label"] == label), key=lambda row: row["id"])
        selected = choices[int(example_rng.integers(len(choices)))]
        reason = "One seeded random validation example, chosen before predictions"
    example_rows.append(selected)

manifest = {
    "schemaVersion": 1, "experimentVersion": EXPERIMENT_VERSION, "classNames": CLASS_NAMES,
    "counts": counts_by_subset, "splitFractionsTarget": [0.70, 0.15, 0.15],
    "splitPolicy": "Fresh deterministic grouped allocation; named examples' entire groups reserved in validation",
    "images": portable_rows(records), "validationProbabilityOrder": "images subset=val in listed order",
}
save_json_once(RUN_DIR / "input_manifest.json", manifest)
save_json_once(RUN_DIR / "class_names.json", CLASS_NAMES)
save_json_once(RUN_DIR / "split_review.json", {"complete": True, "notes": SPLIT_REVIEW_NOTES, "counts": counts_by_subset})
save_json_once(RUN_DIR / "example_selection.json", {"seed": SEED, "subset": "val", "images": portable_rows(example_rows), "selectionBeforePredictions": True})
with zipfile.ZipFile(RUN_DIR / "dataset_split.zip", "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for row in records:
        archive.write(row["path"], row["relativePath"])
print("Frozen split and seven example identities saved:", RUN_DIR)

## Step 7: Review background scenes

**What:** Select appropriate replacement scenes from the 225 collected photographs.

**Why:** A collected scene is not automatically suitable. Prominent people, text, unrelated objects, or mango task images can create misleading examples.

**How:** Inspect every scene contact sheet. Enter only explicitly approved scene IDs and source notes. Review near-identical scenes together; an exact duplicate is not counted as an additional unique scene.

**Expected result:** A nonempty, explicitly approved scene list is saved. Raw backgrounds remain unchanged, and collected/approved counts are both reported.

In [ ]:
%%mangoleaf
save_contact_sheets(background_rows, RUN_DIR / "scene_previews", "Background review: approve suitable scenes only")
SCENE_REVIEW_COMPLETE = False
APPROVED_SCENE_IDS = []
SCENE_REVIEW_NOTES = ""
SCENE_SOURCE_NOTES = {}  # Manually enter origin, permission, and related-scene notes where known.

if not SCENE_REVIEW_COMPLETE or not SCENE_REVIEW_NOTES or not APPROVED_SCENE_IDS:
    raise RuntimeError("Inspect all scene sheets and enter explicitly approved scene IDs and review notes.")
known_scene_ids = {row["id"] for row in background_rows}
if len(APPROVED_SCENE_IDS) != len(set(APPROVED_SCENE_IDS)) or not set(APPROVED_SCENE_IDS).issubset(known_scene_ids):
    raise ValueError("Approved scene IDs must be unique known identities.")
approved_scenes = [row for row in background_rows if row["id"] in set(APPROVED_SCENE_IDS)]
if len({row["decodedRgbSha256"] for row in approved_scenes}) != len(approved_scenes):
    raise ValueError("Approved scenes contain identical decoded images. Select one representative of each exact identity.")
verify_image_rows(approved_scenes)
save_json_once(RUN_DIR / "scene_review.json", {
    "complete": True, "collectedCount": len(background_rows), "approvedCount": len(approved_scenes),
    "approvedIds": APPROVED_SCENE_IDS, "notes": SCENE_REVIEW_NOTES, "sourceNotes": SCENE_SOURCE_NOTES,
})
print("Collected backgrounds:", len(background_rows), "Approved unique scenes:", len(approved_scenes))

## Step 8: Review which photos can receive replacement backgrounds

**What:** Number the photos and suggest which have a mostly white border.

**Why:** Background replacement requires an object that can be separated reliably. The border test is a suggestion, not proof of a good foreground mask. Short photo numbers avoid typing long IDs.

**How:** Run this cell to generate the numbered training and display review sheets. The first run stops until the review is complete. Inspect the sheets in `white_training_numbered_previews` and `white_display_numbered_previews`. Enter the selected numbers in `ELIGIBLE_TRAIN_PHOTO_NUMBERS` and `ELIGIBLE_DISPLAY_PHOTO_NUMBERS`, add review notes, set `WHITE_REVIEW_COMPLETE = True`, and run the cell again. Numbers continue across pages, but each pool starts at 1. Training photo 1 and display photo 1 are separate selections. IDs are filled automatically; photos are not approved automatically. Display examples remain outside training and unchanged during validation scoring.

**Expected result:** Two separate eligibility lists are saved with the original photo IDs. An empty number list selects no photos from that pool. Photos without replacement masks still receive the other training augmentations. Step 9 generates masks for inspection, and Step 10 requires at least one approved training mask for this seven-technique experiment.


In [ ]:
%%mangoleaf
# Edit only this review block after inspecting the numbered sheets.
WHITE_REVIEW_COMPLETE = False
ELIGIBLE_TRAIN_PHOTO_NUMBERS = []
ELIGIBLE_DISPLAY_PHOTO_NUMBERS = []
WHITE_REVIEW_NOTES = ""


def white_border_fraction(row):
    pixels = load_pixels(row["path"]).numpy()
    border = np.concatenate((pixels[:8].reshape(-1, 3), pixels[-8:].reshape(-1, 3), pixels[:, :8].reshape(-1, 3), pixels[:, -8:].reshape(-1, 3)))
    return float(np.mean((border.min(axis=1) >= 225) & (border.max(axis=1) - border.min(axis=1) <= 25)))


def numbered_mask_review_rows(rows):
    """Add short numbers to preview captions without changing real photo IDs."""
    return [
        dict(row, id=f"Photo {number} | " + row["id"])
        for number, row in enumerate(rows, start=1)
    ]


def mask_ids_from_photo_numbers(numbers, rows, pool):
    """Look up exact IDs from the numbers shown in the matching review sheets."""
    if not isinstance(numbers, (list, tuple)) or any(type(number) is not int for number in numbers):
        raise ValueError(pool + " photo numbers must be a list of whole numbers.")
    if len(numbers) != len(set(numbers)):
        raise ValueError(pool + " photo numbers must not repeat.")
    if any(number < 1 or number > len(rows) for number in numbers):
        raise ValueError(pool + " photo numbers must be between 1 and " + str(len(rows)) + ".")

    # Lists start at index 0, while the visible photo numbers start at 1.
    return [rows[number - 1]["id"] for number in numbers]


white_suggestions = [{"id": row["id"], "subset": row["subset"], "whiteBorderFraction": white_border_fraction(row)} for row in train_rows + example_rows]
save_json_once(RUN_DIR / "white_background_suggestions.json", white_suggestions)

# New folders keep the earlier, unnumbered review sheets intact.
save_contact_sheets(
    numbered_mask_review_rows(train_rows),
    RUN_DIR / "white_training_numbered_previews",
    "Training photos: select suitable photo numbers",
)
save_contact_sheets(
    numbered_mask_review_rows(example_rows),
    RUN_DIR / "white_display_numbered_previews",
    "Seven display examples: select suitable photo numbers",
)
print("Numbered training sheets:", RUN_DIR / "white_training_numbered_previews")
print("Numbered display sheets:", RUN_DIR / "white_display_numbered_previews")

if not WHITE_REVIEW_COMPLETE or not WHITE_REVIEW_NOTES.strip():
    raise RuntimeError(
        "Numbered sheets are ready. Inspect them, enter suitable photo numbers and review notes, "
        "set WHITE_REVIEW_COMPLETE = True, then run this cell again."
    )

# These IDs are generated automatically. Step 9 uses the existing ID variables.
ELIGIBLE_TRAIN_MASK_IDS = mask_ids_from_photo_numbers(
    ELIGIBLE_TRAIN_PHOTO_NUMBERS, train_rows, "Training",
)
ELIGIBLE_DISPLAY_MASK_IDS = mask_ids_from_photo_numbers(
    ELIGIBLE_DISPLAY_PHOTO_NUMBERS, example_rows, "Display",
)
save_json_once(RUN_DIR / "white_background_review.json", {
    "complete": True, "trainingIds": ELIGIBLE_TRAIN_MASK_IDS,
    "displayOnlyValidationIds": ELIGIBLE_DISPLAY_MASK_IDS, "notes": WHITE_REVIEW_NOTES,
})
print("Selected training photos:", len(ELIGIBLE_TRAIN_MASK_IDS))
print("Selected display photos:", len(ELIGIBLE_DISPLAY_MASK_IDS))


## Step 9: Generate fresh foreground masks

**What:** Create new masks for the approved eligibility lists.

**Why:** A mask marks which pixels belong to the leaf or fruit. Earlier masks are never reused, and a new mask still needs visual approval.

**How:** Use OpenCV GrabCut with a white-border initialization. The algorithm uses current image colors to separate a probable foreground from the background. It does not verify disease labels or guarantee lesion preservation. Training and display-only masks are saved in separate folders.

**Expected result:** Mask files and original/mask/composite sheets are generated. Inspect edges, holes, and disease patches before approving anything.

In [ ]:
%%mangoleaf
import cv2

MASK_ROOT = RUN_DIR / "masks"
mask_paths = {}
records_by_id = {row["id"]: row for row in records}


def create_foreground_mask(row):
    pixels = np.uint8(np.clip(load_pixels(row["path"]).numpy(), 0, 255))
    white = (pixels.min(axis=2) >= 225) & (pixels.max(axis=2) - pixels.min(axis=2) <= 25)
    labels = np.full(IMAGE_SIZE, cv2.GC_PR_FGD, dtype=np.uint8)
    labels[white] = cv2.GC_PR_BGD
    labels[:2, :] = cv2.GC_BGD
    labels[-2:, :] = cv2.GC_BGD
    labels[:, :2] = cv2.GC_BGD
    labels[:, -2:] = cv2.GC_BGD
    background_model = np.zeros((1, 65), dtype=np.float64)
    foreground_model = np.zeros((1, 65), dtype=np.float64)
    cv2.setRNGSeed(SEED)
    cv2.grabCut(pixels, labels, None, background_model, foreground_model, 5, cv2.GC_INIT_WITH_MASK)
    return np.uint8(np.isin(labels, (cv2.GC_FGD, cv2.GC_PR_FGD))) * 255


def blend_scene(pixels, mask, scene_pixels):
    alpha = np.asarray(mask, dtype=np.float32) / 255.0
    if alpha.ndim == 2:
        alpha = alpha[..., None]
    return np.clip(pixels * alpha + scene_pixels * (1.0 - alpha), 0, 255).astype(np.float32)


for pool, keys in (("training", ELIGIBLE_TRAIN_MASK_IDS), ("display_only", ELIGIBLE_DISPLAY_MASK_IDS)):
    destination = MASK_ROOT / pool
    destination.mkdir(parents=True, exist_ok=True)
    for key in keys:
        row = records_by_id[key]
        mask_path = destination / f"{key}.png"
        if mask_path.exists():
            raise RuntimeError("Fresh mask already exists. Review saved masks; do not regenerate over them.")
        mask = create_foreground_mask(row)
        Image.fromarray(mask).save(mask_path)
        mask_paths[key] = mask_path
        original = load_pixels(row["path"]).numpy()
        scene = load_pixels(approved_scenes[0]["path"]).numpy()
        preview = Image.new("RGB", (672, 252), "white")
        preview.paste(Image.fromarray(np.uint8(original)), (0, 28))
        preview.paste(Image.fromarray(mask).convert("RGB"), (224, 28))
        preview.paste(Image.fromarray(np.uint8(blend_scene(original, mask, scene))), (448, 28))
        ImageDraw.Draw(preview).text((4, 8), f"{pool} {key}: original / fresh mask / composite", fill="black")
        preview.save(destination / f"{key}_preview.png")

save_json_once(RUN_DIR / "mask_inventory.json", {
    "method": "OpenCV GrabCut initialized from a white-border eligibility review; no inherited masks",
    "opencvVersion": cv2.__version__,
    "images": [{"id": key, "relativePath": path.relative_to(RUN_DIR).as_posix(), "sha256": file_sha256(path),
                "use": "training" if key in ELIGIBLE_TRAIN_MASK_IDS else "display_only"} for key, path in mask_paths.items()],
})
print("Inspect every mask and composite in:", MASK_ROOT)

## Step 10: Approve masks and lighting safely

**What:** Record explicit mask approvals and lighting review decisions.

**Why:** A bad cutout can erase disease evidence, and a strong shadow can hide it. Automatic generation is never automatic approval.

**How:** Enter approved training and display mask IDs separately after inspecting every composite. Rejected masks leave their original photos in the dataset. Inspect highlight/shadow examples and confirm that lesions remain interpretable.

**Expected result:** Only approved masks can change training backgrounds. Display-only validation masks remain separate. Lighting decisions are saved before training.

In [ ]:
%%mangoleaf
MASK_REVIEW_COMPLETE = False
APPROVED_TRAIN_MASK_IDS = []
APPROVED_DISPLAY_MASK_IDS = []
MASK_REVIEW_NOTES = ""

if not MASK_REVIEW_COMPLETE or not MASK_REVIEW_NOTES:
    raise RuntimeError("Inspect all masks, then explicitly enter approved IDs and notes.")
if not set(APPROVED_TRAIN_MASK_IDS).issubset(set(ELIGIBLE_TRAIN_MASK_IDS)) or not set(APPROVED_DISPLAY_MASK_IDS).issubset(set(ELIGIBLE_DISPLAY_MASK_IDS)):
    raise ValueError("An approved mask must belong to its corresponding eligible list.")
if not APPROVED_TRAIN_MASK_IDS:
    raise RuntimeError("This seven-technique experiment needs at least one approved training mask for background replacement. Review eligibility rather than forcing a bad mask.")
approved_mask_ids = set(APPROVED_TRAIN_MASK_IDS)
display_mask_ids = set(APPROVED_DISPLAY_MASK_IDS)
save_json_once(RUN_DIR / "mask_review.json", {
    "complete": True, "approvedTrainingIds": APPROVED_TRAIN_MASK_IDS,
    "approvedDisplayOnlyIds": APPROVED_DISPLAY_MASK_IDS, "notes": MASK_REVIEW_NOTES,
})


def stable_seed(*values):
    return int(hashlib.sha256("|".join(map(str, values)).encode("utf-8")).hexdigest()[:8], 16)


def add_local_light(pixels, key, epoch, force=None):
    generator = random.Random(stable_seed(SEED, key, epoch, "local-light"))
    if force is None and generator.random() >= LOCAL_LIGHT_PROBABILITY:
        return np.asarray(pixels, dtype=np.float32)
    sign = force if force is not None else generator.choice((-1, 1))
    amplitude = generator.uniform(*LIGHT_AMPLITUDE) * sign
    radius = generator.uniform(*LIGHT_RADIUS) * min(IMAGE_SIZE)
    center_x = generator.uniform(0, IMAGE_SIZE[1] - 1)
    center_y = generator.uniform(0, IMAGE_SIZE[0] - 1)
    y, x = np.mgrid[:IMAGE_SIZE[0], :IMAGE_SIZE[1]].astype(np.float32)
    patch = np.exp(-((x - center_x) ** 2 + (y - center_y) ** 2) / (2 * radius ** 2))
    return np.clip(np.asarray(pixels, dtype=np.float32) + amplitude * patch[..., None], 0, 255)


lighting_directory = RUN_DIR / "lighting_previews"
lighting_directory.mkdir(exist_ok=True)
for row in example_rows:
    original = load_pixels(row["path"]).numpy()
    preview = Image.new("RGB", (672, 252), "white")
    for index, pixels in enumerate((original, add_local_light(original, row["id"], 0, 1), add_local_light(original, row["id"], 0, -1))):
        preview.paste(Image.fromarray(np.uint8(pixels)), (index * 224, 28))
    ImageDraw.Draw(preview).text((4, 8), row["label"] + ": original / highlight / shadow", fill="black")
    preview.save(lighting_directory / f"{row['id']}.png")

LIGHTING_REVIEW_COMPLETE = False
LIGHTING_REVIEW_NOTES = ""
if not LIGHTING_REVIEW_COMPLETE or not LIGHTING_REVIEW_NOTES:
    raise RuntimeError("Review the seven lighting previews and enter the lighting review decision.")
save_json_once(RUN_DIR / "lighting_review.json", {"complete": True, "notes": LIGHTING_REVIEW_NOTES})

## Step 11: Define the seven training augmentations

**What:** Build the agreed flip, rotation, zoom, shear, brightness, background, and local-light transformations.

**Why:** The new experiment uses the established recipe while generating its own decisions and results.

**How:** The five ordinary transformations run only on training batches. Background replacement applies independently with a 30% chance only to approved training masks. Local lighting has its own 30% chance. Validation and test inputs remain unchanged.

**Expected result:** Training batches remain float32 RGB 0–255. No blur, noise, compression, hue shift, cutout, or erasing is added.

In [ ]:
%%mangoleaf
class RandomShear(tf.keras.layers.Layer):
    def __init__(self, factor=0.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs, training=False):
        if not training:
            return inputs
        shape = tf.shape(inputs)
        count = shape[0]
        center_x = tf.cast(shape[2], tf.float32) / 2.0
        center_y = tf.cast(shape[1], tf.float32) / 2.0
        shear_x = tf.random.uniform([count], -self.factor, self.factor)
        shear_y = tf.random.uniform([count], -self.factor, self.factor)
        one = tf.ones_like(shear_x)
        zero = tf.zeros_like(shear_x)
        transforms = tf.stack((one, shear_x, -shear_x * center_y, shear_y, one, -shear_y * center_x, zero, zero), axis=1)
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs, transforms=transforms, output_shape=shape[1:3],
            interpolation="BILINEAR", fill_mode="NEAREST", fill_value=0.0,
        )


def make_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        tf.keras.layers.RandomRotation(30 / 360, fill_mode="nearest", seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.2, fill_mode="nearest", seed=SEED + 2),
        RandomShear(0.2),
        tf.keras.layers.RandomBrightness(0.2, value_range=(0.0, 255.0), seed=SEED + 3),
    ], name="training_augmentation")


def transformed_training_pixels(row, epoch):
    pixels = load_pixels(row["path"]).numpy()
    chooser = random.Random(stable_seed(SEED, row["id"], epoch, "background"))
    if row["id"] in approved_mask_ids and chooser.random() < BACKGROUND_PROBABILITY:
        scene = approved_scenes[chooser.randrange(len(approved_scenes))]
        mask = np.asarray(Image.open(mask_paths[row["id"]]).convert("L"))
        pixels = blend_scene(pixels, mask, load_pixels(scene["path"]).numpy())
    return add_local_light(pixels, row["id"], epoch)


augmentation = make_augmentation()


def make_dataset(rows, training=False, epoch=0):
    ordered = list(rows)
    if training:
        np.random.default_rng(stable_seed(SEED, epoch, "order")).shuffle(ordered)

    def examples():
        for row in ordered:
            pixels = transformed_training_pixels(row, epoch) if training else load_pixels(row["path"]).numpy()
            yield pixels.astype(np.float32), np.int32(CLASS_NAMES.index(row["label"]))

    dataset = tf.data.Dataset.from_generator(examples, output_signature=(
        tf.TensorSpec((224, 224, 3), tf.float32), tf.TensorSpec((), tf.int32),
    )).batch(BATCH_SIZE)
    if training:
        dataset = dataset.map(lambda pixels, labels: (augmentation(pixels, training=True), labels), num_parallel_calls=1)
    return dataset.prefetch(1)

## Step 12: Freeze the experiment plan and test conversion

**What:** Save the complete reviewed plan and run a tiny export test.

**Why:** A failing converter should be discovered before a 15-epoch GPU run, and settings must not change after training begins.

**How:** Record class order, grouped split, augmentation values, mask/scene review hashes, preprocessing, epoch settings, and actual dependency versions. Export a small model through the configured isolated converter. No dependency package is silently patched or stubbed.

**Expected result:** The frozen plan and conversion preflight exist. If conversion fails, fix the reviewed environment before training; do not alter historical package snapshots.

In [ ]:
%%mangoleaf
plan = {
    "schemaVersion": 1, "experimentVersion": EXPERIMENT_VERSION, "modelVersion": MODEL_VERSION,
    "classNames": CLASS_NAMES, "seed": SEED, "imageSize": list(IMAGE_SIZE), "batchSize": BATCH_SIZE,
    "initialWeights": "ImageNet MobileNetV3Large; no earlier mango weights",
    "splitPolicy": manifest["splitPolicy"], "splitFractionsTarget": [0.70, 0.15, 0.15],
    "stage1": {"epochs": STAGE1_EPOCHS, "learningRate": STAGE1_LEARNING_RATE, "backboneFrozen": True},
    "stage2": {"epochs": STAGE2_EPOCHS, "learningRate": STAGE2_LEARNING_RATE, "batchNormalizationFrozen": True},
    "modelSelection": "Restore minimum validation loss within stage 1; final model is minimum validation loss within stage 2",
    "phashDistance": PHASH_DISTANCE,
    "augmentation": {"flip": "horizontal_and_vertical", "rotationDegrees": 30, "zoom": 0.2, "shear": 0.2, "brightness": 0.2,
                     "backgroundProbability": BACKGROUND_PROBABILITY, "localLightProbability": LOCAL_LIGHT_PROBABILITY,
                     "lightAmplitude": list(LIGHT_AMPLITUDE), "lightRadius": list(LIGHT_RADIUS)},
    "preprocessing": PREPROCESSING, "temperatureRule": "Fit validation NLL only; divide final Dense logits once",
    "testPredictionsRun": False, "backgroundCollected": len(background_rows), "backgroundApproved": len(approved_scenes),
    "validationExamples": [row["id"] for row in example_rows], "runtime": RUNTIME,
}
save_json_once(RUN_DIR / "plan.json", plan)
frozen_paths = ["plan.json", "input_manifest.json", "dataset_split.zip", "mango_inventory.json", "scene_inventory.json", "ood_inventory.json",
                "duplicate_audit.json", "duplicate_review.json", "split_review.json", "example_selection.json", "white_background_review.json",
                "mask_inventory.json", "mask_review.json", "lighting_review.json", "scene_review.json", "runtime.json"]
frozen_paths.extend(path.relative_to(RUN_DIR).as_posix() for path in mask_paths.values())
frozen_paths.extend(path.relative_to(RUN_DIR).as_posix() for path in sorted((RUN_DIR / "runtime_preflight").rglob("*")) if path.is_file())
frozen_artifacts = {relative: {"sha256": file_sha256(safe_child(RUN_DIR, relative)), "bytes": safe_child(RUN_DIR, relative).stat().st_size} for relative in frozen_paths}
save_json_once(RUN_DIR / "frozen_inputs.json", {"configurationHash": object_sha256(plan), "artifacts": frozen_artifacts})

if not Path(CONVERTER_PYTHON).is_file():
    raise RuntimeError("Set CONVERTER_PYTHON to the Python interpreter in the reviewed isolated TensorFlow.js export environment.")
converter_runtime = subprocess.run([CONVERTER_PYTHON, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
(RUN_DIR / "converter_requirements_snapshot.txt").write_text(converter_runtime, encoding="utf-8")


save_json_once(RUN_DIR / "converter_preflight.json", {
    "passed": True, "interpreter": CONVERTER_PYTHON, "record": RUNTIME_REVIEW["record"],
    "noQuantization": True, "browserProbabilityParity": "not measured by this preflight",
})
print("Recorded architecture conversion preflight. The 15-epoch training can begin after the frozen plan review.")

## Step 13: Build the fresh model and calculate class weights

**What:** Load ImageNet features and attach a new seven-class output layer.

**Why:** Transfer learning reuses general visual features, while fresh class weights give underrepresented training classes more influence.

**How:** MobileNetV3-Large receives float32 RGB 0–255 and includes its own preprocessing. Freeze the backbone for head training. Calculate balanced weights from training labels only; validation and test counts do not set them.

**Expected result:** A new model is built, class weights and initial-weight identity are recorded, and the class output order matches the manifest.

In [ ]:
%%mangoleaf
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet", include_preprocessing=True,
)
base_model.trainable = False
inputs = tf.keras.Input(shape=(224, 224, 3), dtype=tf.float32, name="image")
features = base_model(inputs, training=False)
pooled = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(features)
pooled = tf.keras.layers.Dropout(0.2, name="head_dropout")(pooled)
logits = tf.keras.layers.Dense(len(CLASS_NAMES), name="class_logits")(pooled)
outputs = tf.keras.layers.Activation("softmax", name="class_probabilities")(logits)
model = tf.keras.Model(inputs, outputs, name="mangoleaf_v4_4_0_seven_class")

train_labels = np.array([CLASS_NAMES.index(row["label"]) for row in train_rows])
weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=train_labels)
class_weight = {index: float(value) for index, value in enumerate(weights)}
save_json_once(RUN_DIR / "class_weights.json", {CLASS_NAMES[index]: value for index, value in class_weight.items()})

initial_digest = hashlib.sha256()
for values in model.get_weights():
    initial_digest.update(str(values.shape).encode("utf-8"))
    initial_digest.update(values.tobytes())
save_json_once(RUN_DIR / "initial_weights.json", {"sha256": initial_digest.hexdigest(), "source": "ImageNet backbone plus freshly initialized head", "seed": SEED})
print("Training class weights:", {CLASS_NAMES[index]: value for index, value in class_weight.items()})
model.summary()

## Step 14: Train for five plus ten epochs

**What:** Train the new head, then fine-tune the backbone with smaller steps.

**Why:** The two stages preserve the agreed approach while selecting this run's own best weights by validation loss.

**How:** Train five head epochs at 0.001. Restore the best head checkpoint. Train ten fine-tuning epochs at 0.00001 with BatchNormalization frozen. Each epoch log and saved record includes the optimizer's actual learning rate. Final weights come from stage 2's lowest validation loss, not an assumed epoch 14.

**Expected result:** Fifteen recorded epochs, both stage checkpoints, the selected uncalibrated model, and a fitted-state receipt are saved. An interrupted incomplete fit cannot silently restart in the same folder.

In [ ]:
%%mangoleaf
if (RUN_DIR / "training_complete.json").exists() or (RUN_DIR / "training_started.json").exists():
    raise RuntimeError("Training already started in this run. Recover export from a completed fitted state, or create a new run after an interrupted fit.")
frozen = read_json(RUN_DIR / "frozen_inputs.json")
live_settings = {
    "classNames": CLASS_NAMES, "seed": SEED, "imageSize": list(IMAGE_SIZE), "batchSize": BATCH_SIZE,
    "stage1": {"epochs": STAGE1_EPOCHS, "learningRate": STAGE1_LEARNING_RATE, "backboneFrozen": True},
    "stage2": {"epochs": STAGE2_EPOCHS, "learningRate": STAGE2_LEARNING_RATE, "batchNormalizationFrozen": True},
    "augmentation": {"flip": "horizontal_and_vertical", "rotationDegrees": 30, "zoom": 0.2, "shear": 0.2, "brightness": 0.2,
                     "backgroundProbability": BACKGROUND_PROBABILITY, "localLightProbability": LOCAL_LIGHT_PROBABILITY,
                     "lightAmplitude": list(LIGHT_AMPLITUDE), "lightRadius": list(LIGHT_RADIUS)},
}
frozen_plan = read_json(RUN_DIR / "plan.json")
if any(frozen_plan[key] != value for key, value in live_settings.items()) or frozen["configurationHash"] != object_sha256(frozen_plan):
    raise RuntimeError("Training settings changed after plan freeze. Preserve this run and create a new run.")
if portable_rows(records) != read_json(RUN_DIR / "input_manifest.json")["images"]:
    raise RuntimeError("Live image labels, groups, order, or subsets differ from the frozen manifest.")
if approved_mask_ids != set(read_json(RUN_DIR / "mask_review.json")["approvedTrainingIds"]):
    raise RuntimeError("Live training mask approvals differ from the frozen review.")
if {row["id"] for row in approved_scenes} != set(read_json(RUN_DIR / "scene_review.json")["approvedIds"]):
    raise RuntimeError("Live approved scenes differ from the frozen review.")
if {CLASS_NAMES[index]: value for index, value in class_weight.items()} != read_json(RUN_DIR / "class_weights.json"):
    raise RuntimeError("Live class weights differ from the recorded training-only calculation.")
for relative, expected in frozen["artifacts"].items():
    if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
        raise RuntimeError("Frozen review/input changed: " + relative)
verify_image_rows(records)
verify_image_rows(approved_scenes)
save_json_once(RUN_DIR / "training_started.json", {"startedAt": datetime.now(timezone.utc).isoformat(), "configurationHash": frozen["configurationHash"]})


class LearningRateLog(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Read the optimizer's actual value without changing its schedule or steps.
        value = self.model.optimizer.learning_rate
        if callable(value):
            value = value(self.model.optimizer.iterations)
        logs["learning_rate"] = float(tf.keras.backend.get_value(value))


training_history = []
completed_epochs = 0
for stage, epoch_count, learning_rate in ((1, STAGE1_EPOCHS, STAGE1_LEARNING_RATE), (2, STAGE2_EPOCHS, STAGE2_LEARNING_RATE)):
    if stage == 2:
        base_model.trainable = True
        for layer in base_model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    best_loss = math.inf
    best_epoch = None
    checkpoint_path = RUN_DIR / f"stage{stage}_best.weights.h5"
    for stage_epoch in range(epoch_count):
        print(f"Stage {stage}; overall epoch {completed_epochs + 1}/{STAGE1_EPOCHS + STAGE2_EPOCHS}")
        result = model.fit(
            make_dataset(train_rows, training=True, epoch=completed_epochs),
            validation_data=make_dataset(val_rows), epochs=1, class_weight=class_weight,
            callbacks=[LearningRateLog()], verbose=2,
        )
        metrics = {name: float(values[-1]) for name, values in result.history.items()}
        if not all(math.isfinite(value) for value in metrics.values()):
            raise RuntimeError("Training produced a non-finite metric. This incomplete run must not be exported.")
        completed_epochs += 1
        row = {"stage": stage, "epoch": completed_epochs, **metrics}
        training_history.append(row)
        save_json_once(RUN_DIR / f"epoch_{completed_epochs:02d}.json", row)
        if metrics["val_loss"] < best_loss:
            best_loss = metrics["val_loss"]
            best_epoch = completed_epochs
            model.save_weights(checkpoint_path)
    model.load_weights(checkpoint_path)
    save_json_once(RUN_DIR / f"stage{stage}_selection.json", {"stage": stage, "selectedEpoch": best_epoch, "validationLoss": best_loss})

selected_model = model
selected_model.save(RUN_DIR / "model_selected.keras")
save_json_once(RUN_DIR / "training_history.json", training_history)
verify_image_rows(records)
verify_image_rows(approved_scenes)
for relative, expected in frozen["artifacts"].items():
    if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
        raise RuntimeError("A frozen input/review changed during training: " + relative)
fitted_paths = list(frozen["artifacts"]) + ["model_selected.keras", "training_history.json", "stage1_best.weights.h5", "stage2_best.weights.h5", "stage1_selection.json", "stage2_selection.json", "class_weights.json", "initial_weights.json"]
save_json_once(RUN_DIR / "training_state.json", {
    "status": "fitted; calibration/export pending", "configurationHash": frozen["configurationHash"],
    "completedEpochs": completed_epochs, "testPredictionsRun": False,
    "artifacts": {relative: {"sha256": file_sha256(safe_child(RUN_DIR, relative)), "bytes": safe_child(RUN_DIR, relative).stat().st_size} for relative in fitted_paths},
})
print("Selected fine-tuning checkpoint:", read_json(RUN_DIR / "stage2_selection.json"))

## Step 15: Fit temperature using validation only

**What:** Adjust probability confidence without training on test or OOD-report images.

**Why:** Temperature calibration can improve how well confidence reflects validation outcomes. Its value must be fitted again for the fresh model.

**How:** Read the selected model's validation logits, minimize validation negative log likelihood, and divide final Dense weights and biases by that temperature exactly once. Verify saved/reloaded probabilities and preserve validation order.

**Expected result:** model_calibrated.keras, temperature.json, and calibrated validation_probabilities.npy are saved. No final test predictions have run.

In [ ]:
%%mangoleaf
for relative, expected in read_json(RUN_DIR / "training_state.json")["artifacts"].items():
    if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
        raise RuntimeError("Fitted artifact changed before calibration: " + relative)
if (RUN_DIR / "calibration_complete.json").exists():
    calibration_receipt = read_json(RUN_DIR / "calibration_complete.json")
    for relative, expected in calibration_receipt["artifacts"].items():
        if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
            raise RuntimeError("Saved calibration artifact changed: " + relative)
    calibrated_model = tf.keras.models.load_model(RUN_DIR / "model_calibrated.keras", compile=False, safe_mode=True)
    temperature_record = read_json(RUN_DIR / "temperature.json")
else:
    partial_calibration = [name for name in ("model_calibrated.keras", "validation_probabilities.npy", "temperature.json") if (RUN_DIR / name).exists()]
    if partial_calibration:
        raise RuntimeError("Incomplete calibration artifacts exist. Preserve them and resolve the failed attempt before exporting; never overwrite them silently: " + str(partial_calibration))
    logits_model = tf.keras.Model(selected_model.inputs, selected_model.get_layer("class_logits").output)
    validation_logits = predict_records(logits_model, val_rows).astype(np.float64)
    validation_labels = np.array([CLASS_NAMES.index(row["label"]) for row in val_rows])

    def validation_nll(temperature):
        scaled = validation_logits / temperature
        shifted = scaled - scaled.max(axis=1, keepdims=True)
        log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        return float(-np.mean(log_probabilities[np.arange(len(validation_labels)), validation_labels]))

    calibration = minimize_scalar(validation_nll, bounds=(0.25, 20.0), method="bounded")
    if not calibration.success or not math.isfinite(calibration.fun):
        raise RuntimeError("Validation-only temperature fit failed.")
    # Keep temperature 1.0 if the bounded fit does not improve validation loss.
    temperature = float(calibration.x) if calibration.fun < validation_nll(1.0) else 1.0
    original_probabilities = predict_records(selected_model, val_rows)
    calibrated_model = tf.keras.models.clone_model(selected_model)
    calibrated_model.set_weights(selected_model.get_weights())
    head = calibrated_model.get_layer("class_logits")
    head_weights, head_bias = head.get_weights()
    head.set_weights([head_weights / temperature, head_bias / temperature])
    probabilities = predict_records(calibrated_model, val_rows)
    if not np.array_equal(original_probabilities.argmax(axis=1), probabilities.argmax(axis=1)):
        raise RuntimeError("Temperature folding changed a validation class prediction.")
    calibrated_model.save(RUN_DIR / "model_calibrated.keras")
    np.save(RUN_DIR / "validation_probabilities.npy", probabilities)
    reloaded = tf.keras.models.load_model(RUN_DIR / "model_calibrated.keras", compile=False, safe_mode=True)
    np.testing.assert_allclose(probabilities, predict_records(reloaded, val_rows), rtol=1e-4, atol=1e-5)
    temperature_record = {
        "temperature": temperature, "temperatureFoldedIntoWeights": True, "foldCount": 1,
        "fitSubset": "val", "validationCount": len(val_rows), "beforeNll": validation_nll(1.0), "afterNll": validation_nll(temperature),
        "searchBounds": [0.25, 20.0], "keepUnitTemperatureUnlessImproved": True,
        "validationProbabilityOrder": "input_manifest.json images subset=val in listed order",
    }
    save_json_once(RUN_DIR / "temperature.json", temperature_record)
    calibration_paths = ["model_calibrated.keras", "validation_probabilities.npy", "temperature.json"]
    save_json_once(RUN_DIR / "calibration_complete.json", {
        "configurationHash": read_json(RUN_DIR / "frozen_inputs.json")["configurationHash"],
        "artifacts": {relative: {"sha256": file_sha256(RUN_DIR / relative), "bytes": (RUN_DIR / relative).stat().st_size} for relative in calibration_paths},
    })
print("New validation-fitted temperature:", temperature_record["temperature"])

## Step 16: Export the calibrated model without quantization

**What:** Convert the fresh calibrated model to TensorFlow.js assets.

**Why:** The browser needs its own graph and weight files with the exact training preprocessing and class order.

**How:** Use the reviewed isolated converter and preserve each failed attempt in a unique directory. Verify graph/shard integrity and keep temperature folded once. Candidate metadata deliberately excludes rejection thresholds, which belong to the separate threshold-fitting workflow.

**Expected result:** tfjs_model contains an unquantized graph, all shards, and candidate model metadata. Actual browser prediction parity is a separate adoption check, not claimed by this conversion.

In [ ]:
%%mangoleaf
def convert_saved_model(saved_model_path, export_path):
    command = [CONVERTER_COMMAND, "--input_format=tf_saved_model",
               "--output_format=tfjs_graph_model", "--signature_name=serving_default", "--saved_model_tags=serve",
               str(saved_model_path), str(export_path)]
    result = subprocess.run(command, capture_output=True, text=True, env=CONVERTER_ENVIRONMENT)
    if result.returncode != 0:
        raise RuntimeError("Converter failed. Review this environment instead of changing model behavior.\n" + result.stderr[-3500:])
    graph = read_json(export_path / "model.json")
    if graph.get("format") != "graph-model" or not graph.get("weightsManifest"):
        raise RuntimeError("Converter did not produce a TensorFlow.js graph model.")
    for group in graph["weightsManifest"]:
        for relative in group["paths"]:
            safe_child(export_path, relative).stat()
        if any(weight.get("quantization") is not None for weight in group["weights"]):
            raise RuntimeError("This experiment requires an unquantized export.")

configuration = {
    "modelVersion": MODEL_VERSION, "experimentVersion": EXPERIMENT_VERSION, "status": "experimental candidate; thresholds not fitted",
    "classNames": CLASS_NAMES, "classes": CLASS_NAMES, "inputSize": [224, 224], "inputRange": [0, 255], "inputDtype": "float32",
    "preprocessing": PREPROCESSING, "temperature": temperature_record["temperature"], "temperatureFoldedIntoWeights": True,
    "quantized": False, "browserParityStatus": "pending actual browser comparison before adoption",
}
destination = RUN_DIR / "tfjs_model"
if (RUN_DIR / "export_complete.json").exists():
    previous_export = read_json(RUN_DIR / "export_complete.json")
    if previous_export["calibratedModelSha256"] != file_sha256(RUN_DIR / "model_calibrated.keras"):
        raise RuntimeError("Calibrated model differs from its completed export.")
    for relative, expected in previous_export["artifacts"].items():
        if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
            raise RuntimeError("Completed export changed: " + relative)
else:
    if destination.exists():
        raise RuntimeError("Unsealed final TFJS output exists. Preserve this run and use a new reviewed export recovery; never overwrite it.")
    export_attempt = RUN_DIR / "export_attempts" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
    export_attempt.mkdir(parents=True, exist_ok=False)
    saved_model_path = export_attempt / "saved_model"
    exported_path = export_attempt / "tfjs_model"
    calibrated_model.export(saved_model_path, verbose=False)
    convert_saved_model(saved_model_path, exported_path)
    save_json_once(exported_path / "model_config.json", configuration)
    shutil.copytree(exported_path, destination)
    save_json_once(RUN_DIR / "export_complete.json", {
        "format": "graph-model", "quantized": False, "attempt": export_attempt.relative_to(RUN_DIR).as_posix(),
        "calibratedModelSha256": file_sha256(RUN_DIR / "model_calibrated.keras"),
        "artifacts": {path.relative_to(RUN_DIR).as_posix(): {"sha256": file_sha256(path), "bytes": path.stat().st_size} for path in sorted(destination.rglob("*")) if path.is_file()},
    })
print("Candidate export:", destination)

## Step 17: Review illustration masks and show all seven augmentation categories

**What:** The same seven validation photos illustrate every augmentation category. Horizontal and vertical flips, highlight and shadow, and background with light receive separate panels.

**Why:** A small preview cannot show every technique clearly. Separate class sheets make the effects easier to inspect. Validation photos remain outside training, and their illustrated copies never enter validation scoring.

**How:** Fresh masks are proposed only for these illustration photos. The contact sheet and masks require a manual `approved` or `skip` decision in the generated CSV. Approval needs a white background, an intact leaf or fruit, and visible disease details. An unsuitable mask remains skipped; background panels then state that the replacement was not applied. This review is independent of the training-mask review.

**Expected result:** Seven readable class sheets, a shared-source overview, and a manifest linking every illustration to its original image hash. These are new demonstration draws, rather than records of exact tensors used during training.


In [ ]:
%%mangoleaf
import csv
import hashlib
import json
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf


FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
PREVIEW_DIR = RUN_DIR / "illustration_masks"
PREVIEW_DIR.mkdir(exist_ok=True)
PREVIEW_REVIEW = PREVIEW_DIR / "review.csv"
FIGURE_PREFIX = "V4.4.0 Seven Class"


def figure_image_path(row):
    return Path(row["path"])


def figure_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def illustration_mask(pixels):
    """Remove only bright, nearly neutral pixels connected to the border.

    This proposal is deliberately conservative. Manual review decides whether
    it preserves the specimen; a mask is never approved by this function.
    """
    pixels = np.asarray(pixels, dtype=np.float32)
    neutral = np.max(pixels, axis=-1) - np.min(pixels, axis=-1) < 30
    candidate = (np.min(pixels, axis=-1) > 220) & neutral
    height, width = candidate.shape
    background = np.zeros((height, width), dtype=bool)
    pending = deque()

    for y in range(height):
        for x in (0, width - 1):
            if candidate[y, x] and not background[y, x]:
                background[y, x] = True
                pending.append((y, x))
    for x in range(width):
        for y in (0, height - 1):
            if candidate[y, x] and not background[y, x]:
                background[y, x] = True
                pending.append((y, x))

    while pending:
        y, x = pending.popleft()
        for next_y, next_x in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
            if 0 <= next_y < height and 0 <= next_x < width:
                if candidate[next_y, next_x] and not background[next_y, next_x]:
                    background[next_y, next_x] = True
                    pending.append((next_y, next_x))

    return (~background).astype(np.float32)


if len(example_rows) != 7 or {row["label"] for row in example_rows} != set(CLASS_NAMES):
    raise ValueError("Illustrations require one frozen validation photo for each class.")

proposal_rows = []
proposal_figure, proposal_axes = plt.subplots(7, 3, figsize=(12, 23))

for index, row in enumerate(example_rows):
    pixels = np.asarray(load_pixels(figure_image_path(row)), dtype=np.float32)
    mask = illustration_mask(pixels)
    mask_path = PREVIEW_DIR / (row["label"] + ".npy")

    if mask_path.exists():
        if not np.array_equal(np.load(mask_path, allow_pickle=False), mask):
            raise RuntimeError("An illustration mask changed; preserve the run and review a new run.")
    else:
        np.save(mask_path, mask, allow_pickle=False)

    proposal_rows.append({
        "label": row["label"],
        "imageSha256": figure_file_hash(figure_image_path(row)),
        "maskSha256": figure_file_hash(mask_path),
        "status": "pending",
        "notes": "",
    })

    views = (pixels / 255, mask, pixels * mask[..., None] / 255)
    headings = (row["label"].replace("_", " "), "Proposed mask", "Retained specimen")
    for axis, view, heading in zip(proposal_axes[index], views, headings):
        axis.imshow(view, cmap="gray" if view.ndim == 2 else None, vmin=0, vmax=1)
        axis.set_title(heading)
        axis.axis("off")

proposal_figure.suptitle(FIGURE_PREFIX + " — illustration-only mask proposals", fontsize=16)
proposal_figure.tight_layout(rect=(0, 0, 1, 0.98))
proposal_figure.savefig(PREVIEW_DIR / "contact_sheet.png", dpi=150)
plt.show()
plt.close(proposal_figure)

if not PREVIEW_REVIEW.exists():
    with PREVIEW_REVIEW.open("x", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(proposal_rows[0]))
        writer.writeheader()
        writer.writerows(proposal_rows)

with PREVIEW_REVIEW.open(encoding="utf-8", newline="") as handle:
    illustration_reviews = list(csv.DictReader(handle))

if len(illustration_reviews) != 7 or {row["label"] for row in illustration_reviews} != set(CLASS_NAMES):
    raise ValueError("The illustration review needs exactly seven unique class rows.")

expected_proposals = {row["label"]: row for row in proposal_rows}
for review in illustration_reviews:
    proposal = expected_proposals[review["label"]]
    if any(review[key] != proposal[key] for key in ("imageSha256", "maskSha256")):
        raise ValueError("A reviewed illustration identity changed.")
    if review["status"] not in {"approved", "skip"} or not review["notes"].strip():
        raise RuntimeError("Illustration masks need approved/skip decisions and review notes in: " + str(PREVIEW_REVIEW))

print("Illustration review complete. Validation membership and scoring remain unchanged.")


In [ ]:
%%mangoleaf
def preview_shear(pixels, shear):
    """Apply the same centered projective shear category used in training."""
    height, width = pixels.shape[:2]
    transform = tf.constant([[1.0, shear, -shear * (height - 1) / 2,
                              0.0, 1.0, 0.0, 0.0, 0.0]], dtype=tf.float32)
    output = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.convert_to_tensor(pixels[None], dtype=tf.float32),
        transforms=transform,
        output_shape=tf.constant([height, width], dtype=tf.int32),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0,
    )
    return output[0].numpy()


def preview_light(pixels, sign):
    """Show a fixed highlight/shadow within the training amplitude/radius range."""
    height, width = pixels.shape[:2]
    y, x = np.mgrid[:height, :width]
    radius = 0.27 * min(height, width)
    bump = np.exp(-((x - 0.55 * width) ** 2 + (y - 0.45 * height) ** 2) / (2 * radius ** 2))
    return np.clip(pixels + sign * 30.0 * bump[..., None], 0, 255)


review_by_label = {row["label"]: row for row in illustration_reviews}
scene = np.asarray(load_pixels(Path(approved_scenes[0]["path"])), dtype=np.float32) if approved_scenes else None
figure_manifest = {
    "titlePrefix": FIGURE_PREFIX,
    "status": "experimental illustrations; no evaluation claim",
    "illustrationReviewSha256": figure_file_hash(PREVIEW_REVIEW),
    "sceneSha256": figure_file_hash(Path(approved_scenes[0]["path"])) if approved_scenes else None,
    "augmentationCategories": ["flip", "rotation", "zoom", "shear", "global brightness", "background replacement", "localized light"],
    "examples": [],
}
overview, overview_axes = plt.subplots(7, 2, figsize=(10, 23))

for index, row in enumerate(example_rows):
    pixels = np.asarray(load_pixels(figure_image_path(row)), dtype=np.float32)
    review = review_by_label[row["label"]]
    mask = np.load(PREVIEW_DIR / (row["label"] + ".npy"), allow_pickle=False)
    can_replace = review["status"] == "approved" and scene is not None
    replacement = pixels * mask[..., None] + scene * (1 - mask[..., None]) if can_replace else pixels

    # Separate panels isolate a category. Training still draws them randomly.
    rotated = tf.keras.layers.RandomRotation(30 / 360, seed=42)(pixels, training=True).numpy()
    zoomed = tf.keras.layers.RandomZoom(0.2, seed=42)(pixels, training=True).numpy()
    brightened = tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=42)(pixels, training=True).numpy()
    views = [
        ("Original", pixels),
        ("Horizontal flip", np.fliplr(pixels)),
        ("Vertical flip", np.flipud(pixels)),
        ("Rotation (range ±30°)", rotated),
        ("Zoom (factor 0.2)", zoomed),
        ("Shear (example +0.2)", preview_shear(pixels, 0.2)),
        ("Global brightness (factor 0.2)", brightened),
        ("Localized highlight", preview_light(pixels, 1)),
        ("Localized shadow", preview_light(pixels, -1)),
        ("Background replacement" if can_replace else "Background: not applied", replacement),
        ("Background + localized light" if can_replace else "Background + light: not applied", preview_light(replacement, 1) if can_replace else pixels),
    ]
    sheet, axes = plt.subplots(3, 4, figsize=(16, 12))
    for axis, (heading, view) in zip(axes.flat, views):
        axis.imshow(np.clip(view / 255, 0, 1))
        axis.set_title(heading, fontsize=12)
        axis.axis("off")
    axes.flat[-1].axis("off")
    axes.flat[-1].text(0, 0.75, "Illustration only.\nTraining uses random draws.\nBackground and light each\nhave independent 30% chances.", fontsize=12)
    sheet.suptitle(FIGURE_PREFIX + " — " + row["label"].replace("_", " ") + " augmentation", fontsize=18)
    sheet.tight_layout(rect=(0, 0, 1, 0.96))
    filename = "augmentation_" + row["label"] + ".png"
    sheet.savefig(FIGURE_DIR / filename, dpi=160)
    plt.show()
    plt.close(sheet)

    for axis, (heading, view) in zip(overview_axes[index], (views[0], views[-1])):
        axis.imshow(np.clip(view / 255, 0, 1))
        axis.set_title(row["label"].replace("_", " ") + " — " + heading, fontsize=11)
        axis.axis("off")
    figure_manifest["examples"].append({
        "className": row["label"],
        "sourceName": figure_image_path(row).name,
        "sourceSha256": figure_file_hash(figure_image_path(row)),
        "illustrationMaskStatus": review["status"],
        "backgroundApplied": can_replace,
        "augmentationFigure": filename,
    })

overview.suptitle(FIGURE_PREFIX + " — shared augmentation sources", fontsize=18)
overview.tight_layout(rect=(0, 0, 1, 0.98))
overview.savefig(FIGURE_DIR / "augmentation_preview.png", dpi=160)
plt.show()
plt.close(overview)


## Step 18: Inspect connected Grad-CAM maps for the same seven photos

**What:** A heatmap shows which final backbone regions influence the true class score. Each row also states the predicted class and calibrated confidence.

**Why:** A disease heatmap needs a real gradient path through the trained head. A wrong prediction remains visible; the random class examples are not filtered to hide errors. Heatmaps do not establish classification accuracy or prove that every lesion caused the prediction.

**How:** The backbone first computes its feature map. GradientTape watches that map while the actual pooling, dropout, and linear classifier compute the true class logit. Pooling the gradients creates the spatial map. The calibrated export supplies the separately displayed prediction. A missing or non-finite gradient stops the step. An all-zero map is labeled honestly.

**Expected result:** `grad_cam.png` and recorded true classes, predictions, confidence, target layers, and source hashes. The source photos exactly match the augmentation examples.


In [ ]:
%%mangoleaf
def connected_grad_cam(model, batch, target_index):
    """Watch the feature tensor, then compute the head inside the tape."""
    backbone = next((layer for layer in model.layers if isinstance(layer, tf.keras.Model)), None)
    if backbone is None:
        raise ValueError("A directly connected MobileNet backbone is required.")
    head_layers = model.layers[model.layers.index(backbone) + 1:]
    dense = next((layer for layer in head_layers if isinstance(layer, tf.keras.layers.Dense)), None)
    if dense is None or dense.get_config()["activation"] != "linear":
        raise ValueError("Grad-CAM requires the actual linear logits layer.")

    features = backbone(batch, training=False)
    if features.shape.rank != 4:
        raise ValueError("Grad-CAM requires a spatial backbone output.")

    with tf.GradientTape() as tape:
        tape.watch(features)
        output = features
        for layer in head_layers:
            output = layer(output, training=False)
            if layer is dense:
                logits = output
                break
        score = logits[:, target_index]

    gradients = tape.gradient(score, features)
    if gradients is None or not bool(tf.reduce_all(tf.math.is_finite(gradients))):
        raise RuntimeError("Grad-CAM gradients are missing or non-finite.")
    weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmap = tf.reduce_sum(features[0] * weights[0], axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    maximum = float(tf.reduce_max(heatmap))
    if maximum > 0:
        heatmap = heatmap / maximum
    resized = tf.image.resize(heatmap[..., None], batch.shape[1:3], method="bilinear")[..., 0]
    return resized.numpy(), maximum > 0, backbone.name, dense.name


grad_figure, grad_axes = plt.subplots(7, 3, figsize=(13, 25))
for index, row in enumerate(example_rows):
    pixels = np.asarray(load_pixels(figure_image_path(row)), dtype=np.float32)
    batch = tf.convert_to_tensor(pixels[None], dtype=tf.float32)
    probabilities = np.asarray(calibrated_model(batch, training=False))[0]
    predicted_index = int(np.argmax(probabilities))
    true_index = CLASS_NAMES.index(row["label"])
    heatmap, has_signal, backbone_name, logits_name = connected_grad_cam(selected_model, batch, true_index)
    colored = plt.colormaps["jet"](heatmap)[..., :3]
    overlay = np.clip(0.65 * pixels / 255 + 0.35 * colored, 0, 1) if has_signal else pixels / 255

    prediction_title = "Predicted: " + CLASS_NAMES[predicted_index].replace("_", " ") + f"\nConfidence: {probabilities[predicted_index]:.1%}"
    headings = ("True: " + row["label"].replace("_", " "), "True-class map" if has_signal else "No positive Grad-CAM signal", prediction_title)
    for axis, view, heading in zip(grad_axes[index], (pixels / 255, heatmap, overlay), headings):
        axis.imshow(view, cmap="jet" if view.ndim == 2 else None, vmin=0, vmax=1)
        axis.set_title(heading, fontsize=11)
        axis.axis("off")

    figure_manifest["examples"][index].update({
        "gradCamTargetClass": row["label"],
        "predictedClass": CLASS_NAMES[predicted_index],
        "calibratedConfidence": float(probabilities[predicted_index]),
        "backboneLayer": backbone_name,
        "logitsLayer": logits_name,
        "hasPositiveHeatmap": has_signal,
    })

grad_figure.suptitle(FIGURE_PREFIX + " — validation illustration Grad-CAM", fontsize=18)
grad_figure.tight_layout(rect=(0, 0, 1, 0.98))
grad_figure.savefig(FIGURE_DIR / "grad_cam.png", dpi=160)
plt.show()
plt.close(grad_figure)


## Step 19: Save separate loss and accuracy curves

**What:** Loss and accuracy appear in two independent figures, each showing training and validation across the actual fifteen epochs.

**Why:** Separate figures are easier to read and reuse in a report. The dotted marker identifies the switch from head training to fine-tuning; it is not an evaluation score.

**How:** Read the saved epoch history without smoothing or inventing values. Record the figure hashes and the illustration source manifest alongside the new run.

**Expected result:** `training_loss.png`, `training_accuracy.png`, and `figure_manifest.json`, all associated with this fresh experimental run.


In [ ]:
%%mangoleaf
epochs = [row["epoch"] for row in training_history]
if epochs != list(range(1, 16)):
    raise ValueError("The figure history must contain the actual five plus ten training epochs.")

for key, validation_key, label, filename in (
    ("loss", "val_loss", "Cross-entropy loss", "training_loss.png"),
    ("accuracy", "val_accuracy", "Accuracy", "training_accuracy.png"),
):
    curve, axis = plt.subplots(figsize=(10, 6))
    axis.plot(epochs, [row[key] for row in training_history], label="Training")
    axis.plot(epochs, [row[validation_key] for row in training_history], label="Validation")
    axis.axvline(5.5, color="gray", linestyle=":", label="Fine-tuning starts")
    axis.set(title=FIGURE_PREFIX + " — " + label.lower(), xlabel="Epoch", ylabel=label)
    axis.set_xticks(epochs)
    axis.grid(alpha=0.2)
    axis.legend()
    curve.tight_layout()
    curve.savefig(FIGURE_DIR / filename, dpi=180)
    plt.show()
    plt.close(curve)

figure_manifest["files"] = {
    path.name: figure_file_hash(path)
    for path in sorted(FIGURE_DIR.glob("*.png"))
}
manifest_path = FIGURE_DIR / "figure_manifest.json"
with manifest_path.open("x", encoding="utf-8") as handle:
    json.dump(figure_manifest, handle, indent=2, allow_nan=False)
    handle.write("\n")
print("Separate curves and illustration provenance saved in:", FIGURE_DIR)


## Step 20: Seal the training run for threshold fitting

**What:** Write a completion receipt covering the frozen inputs and exported artifacts.

**Why:** The threshold notebook must read one exact finished training run and verify its hashes before fitting any rejection settings.

**How:** Hash every required artifact, identify the newly selected checkpoint and temperature, and record that final test predictions and browser parity are still pending. Keep all earlier notebooks and adopted app assets unchanged.

**Expected result:** training_complete.json is saved. Copy this exact run path into the new threshold-fitting notebook; do not point it at an earlier experiment.

In [ ]:
%%mangoleaf
if (RUN_DIR / "training_complete.json").exists():
    raise RuntimeError("This run is already sealed. Read its completion receipt without overwriting it.")
fitted = read_json(RUN_DIR / "training_state.json")
if fitted["completedEpochs"] != STAGE1_EPOCHS + STAGE2_EPOCHS:
    raise RuntimeError("The agreed 15 epochs did not complete.")
for receipt_name in ("training_state.json", "calibration_complete.json", "export_complete.json"):
    evidence_receipt = read_json(RUN_DIR / receipt_name)
    for relative, expected in evidence_receipt["artifacts"].items():
        if file_sha256(safe_child(RUN_DIR, relative)) != expected["sha256"]:
            raise RuntimeError("A previously sealed artifact changed before completion: " + relative)
required = [
    "input_manifest.json", "dataset_split.zip", "class_names.json", "plan.json", "runtime.json", "requirements_snapshot.txt",
    "mango_inventory.json", "scene_inventory.json", "ood_inventory.json", "example_selection.json",
    "model_selected.keras", "model_calibrated.keras", "temperature.json", "validation_probabilities.npy", "training_history.json",
    "frozen_inputs.json", "training_state.json", "calibration_complete.json", "export_complete.json",
    "converter_requirements_snapshot.txt", "duplicate_audit.json", "duplicate_review.json", "scene_review.json",
    "mask_inventory.json", "mask_review.json", "white_background_review.json", "lighting_review.json",
    "class_weights.json", "initial_weights.json", "reviewed_images.json", "stage1_selection.json", "stage2_selection.json",
    "stage1_best.weights.h5", "stage2_best.weights.h5", "training_started.json", "split_review.json", "converter_preflight.json",
]
required.extend(read_json(RUN_DIR / "frozen_inputs.json")["artifacts"])
required.extend(f"epoch_{epoch:02d}.json" for epoch in range(1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1))
required.extend(path.relative_to(RUN_DIR).as_posix() for path in sorted((RUN_DIR / "tfjs_model").rglob("*")) if path.is_file())
required.extend(path.relative_to(RUN_DIR).as_posix() for directory in ("runtime_preflight", "figures", "illustration_masks")
                for path in sorted((RUN_DIR / directory).rglob("*")) if path.is_file())
receipt = {
    "schemaVersion": 1, "experimentVersion": EXPERIMENT_VERSION, "modelVersion": MODEL_VERSION,
    "status": "training/export complete; experimental candidate; threshold fitting and browser evaluation pending",
    "completedAt": datetime.now(timezone.utc).isoformat(), "classNames": CLASS_NAMES,
    "configurationHash": fitted["configurationHash"], "inputManifestSha256": file_sha256(RUN_DIR / "input_manifest.json"),
    "calibratedTemperature": temperature_record["temperature"], "temperatureFoldedIntoWeights": True, "foldedTemperature": True,
    "testPredictionsRun": False, "browserParityPassed": False, "completedEpochs": fitted["completedEpochs"],
    "selectedCheckpoint": read_json(RUN_DIR / "stage2_selection.json"),
    "artifacts": {relative: {"sha256": file_sha256(safe_child(RUN_DIR, relative)), "bytes": safe_child(RUN_DIR, relative).stat().st_size} for relative in required},
}
save_json_once(RUN_DIR / "training_complete.json", receipt)
print("Finished standalone training run:", RUN_DIR)
print("Next: new threshold-fitting notebook, using this exact run and the separate OOD assignments.")